# Phase 3: Poisoning, Retention and Legitimate Adaptation in Federated Continual Intrusion Detection
**One notebook, three parts. Each part saves its results separately.** Settings: Internet ON. Accelerator: **GPU recommended (T4)** (Part B can run on CPU or GPU; see B-2).

| Part | Task (Phase 3 requirement) | Results folder | Download |
|---|---|---|---|
| A | (2) Dataset and preprocessing: rebuild the data, verify every file against the repository's pinned hashes, build one training package per seed | `/kaggle/working/outputs/part1` | `phase3-part1-outputs.zip` |
| B | (1) Implementation, (3) baselines, (6) ablations: run the whole experiment grid in parallel (own implementation, `p3lib.py`) | `/kaggle/working/outputs/part2` | `phase3-part2-outputs.zip` |
| C | (4) graphs, (5) comparison with existing research, (7) analysis and discussion: all tables and figures from the saved runs, no training | `/kaggle/working/outputs/part3` | `phase3-part3-outputs.zip` |

Rules kept throughout: development bank only (no final-test data is read), 3 allowed development ownership seeds (101-103) x 4 training replicates = 12 paired units (the count planned in Phase 2), results are reported as measured (never tuned away), confirmatory tests follow the Phase 2 plan (paired Wilcoxon, Holm).
Every long step is resumable. Run cells top to bottom; if the session restarts, run the A-1 helper cell again and continue.

# PART A: environment, data and per-seed training packages

In [1]:
# 1. Config and helpers
import os, sys, subprocess, time, json, hashlib, shutil, platform
W = "/kaggle/working"
R = f"{W}/repo"                      # clone of the project repository
VENV = f"{W}/venv312"
VPY = f"{VENV}/bin/python"
STATE = f"{W}/state"
OUT1, OUT2, OUT3 = (f"{W}/outputs/part{i}" for i in (1, 2, 3))   # results are kept separate per part
for d in (STATE, OUT1, OUT2, OUT3): os.makedirs(d, exist_ok=True)
REPO_URL = "https://github.com/ahmedbilalazhar/cl-ids-adversarial.git"
EXPECTED_COMMIT = "571b5f8"          # commit the plan was written against
DATA_URL = "https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CICIDS2017_improved.zip"
DATA_SHA256 = "97fdb91d339e2d8cf5627f981b831e5e7e400b981c58181c451a38fd03c48883"
DATA_BYTES = 343549013
DEST = "kaggle-reproduction"         # fresh output name: existing result folders are immutable

def log(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)

def run(cmd, cwd=R, logfile=None, check=True):
    """Run a command, stream its output live, stop the notebook on failure."""
    log("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    f = open(logfile, "a") if logfile else None
    for line in p.stdout:
        print(line, end="")
        if f: f.write(line)
    p.wait()
    if f: f.close()
    log(f"exit {p.returncode} after {time.time()-t0:.0f}s")
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")
    return p.returncode

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
    return h.hexdigest()

def done(name): return os.path.exists(f"{STATE}/{name}.done")
def mark(name): open(f"{STATE}/{name}.done", "w").write(time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))

In [2]:
# 2. Environment report and repository clone
log("Python", sys.version.split()[0], "|", platform.platform())
log("CPU cores", os.cpu_count())
try:
    import psutil; log("RAM GB", round(psutil.virtual_memory().total / 1e9, 1))
except Exception as e: log("psutil unavailable", e)
log("Free disk GB", round(shutil.disk_usage(W).free / 1e9, 1))
if not os.path.isdir(f"{R}/.git"):
    run(f"git clone {REPO_URL} {R}", cwd=W)
head = subprocess.run("git log --oneline -1", shell=True, cwd=R, capture_output=True, text=True).stdout.strip()
log("Repository HEAD:", head)
if not head.startswith(EXPECTED_COMMIT):
    print("WARNING: HEAD differs from the commit this plan was written for. Review the new commits before continuing.")

06:59:55 Python 3.13.16 | Linux-6.18.55+-x86_64-with-glibc2.39
06:59:55 CPU cores 4
06:59:55 RAM GB 33.7
06:59:55 Free disk GB 20.9
06:59:55 $ git clone https://github.com/ahmedbilalazhar/cl-ids-adversarial.git /kaggle/working/repo
Cloning into '/kaggle/working/repo'...
Updating files:  68% (6630/9629)
Updating files:  69% (6645/9629)
Updating files:  70% (6741/9629)
Updating files:  71% (6837/9629)
Updating files:  72% (6933/9629)
Updating files:  73% (7030/9629)
Updating files:  74% (7126/9629)
Updating files:  75% (7222/9629)
Updating files:  76% (7319/9629)
Updating files:  77% (7415/9629)
Updating files:  78% (7511/9629)
Updating files:  79% (7607/9629)
Updating files:  80% (7704/9629)
Updating files:  81% (7800/9629)
Updating files:  82% (7896/9629)
Updating files:  83% (7993/9629)
Updating files:  84% (8089/9629)
Updating files:  85% (8185/9629)
Updating files:  86% (8281/9629)
Updating files:  87% (8378/9629)
Updating files:  88% (8474/9629)
Updating files:  89% (8570/9629)
Upd

In [3]:
# 3. Python 3.12 environment with the repository's exact pins (reuses an existing valid environment)
WANT = ["2.3.5", "3.0.1", "1.18.1", "1.9.1", "2.7.0+cpu"]
def venv_versions():
    if not os.path.exists(VPY): return None
    r = subprocess.run([VPY, "-c", "import sys,numpy,pandas,scipy,sklearn,torch;print(sys.version.split()[0],numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,torch.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip().split() if r.returncode == 0 else None
v = venv_versions()
if not v or v[1:] != WANT:
    run("pip -q install uv", cwd=W)
    run(f"uv venv --clear --python 3.12 {VENV}", cwd=W)
    run(f"uv pip install --python {VPY} -r {R}/docs/phase3-plan/tools/audit-requirements.txt", cwd=W)
    run(f"uv pip install --python {VPY} -r {R}/configs/phase3/requirements-reference-cpu.lock --extra-index-url https://download.pytorch.org/whl/cpu", cwd=W)
    v = venv_versions()
log("Versions:", v)
assert v and v[1:] == WANT, "Package versions differ from the repository pins"

07:00:06 $ pip -q install uv
07:00:10 exit 0 after 4s
07:00:10 $ uv venv --clear --python 3.12 /kaggle/working/venv312
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: venv312
Activate with: source venv312/bin/activate
07:00:10 exit 0 after 0s
07:00:10 $ uv pip install --python /kaggle/working/venv312/bin/python -r /kaggle/working/repo/docs/phase3-plan/tools/audit-requirements.txt
Using Python 3.12.3 environment at: venv312
Resolved 18 packages in 212ms
Prepared 18 packages in 1.14s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 496ms
 + cloudpickle==3.1.2
 + contourpy==1.4.0
 + cycler==0.12.1
 + fonttools==4.67.0
 + joblib==1.6.0
 + kiwisolver==1.5.1
 + matplotlib==3.11.2
 + narwhals==2.27.1
 + numpy==2.3.5
 + packaging==26.3
 + pandas==3.0.1
 +

In [4]:
# 4. Download the corrected dataset and verify size and SHA-256
D = f"{R}/data/raw/phase3-feasibility-corrected"; os.makedirs(D, exist_ok=True)
Z = f"{D}/CICIDS2017_improved.zip"
if not (os.path.exists(Z) and os.path.getsize(Z) == DATA_BYTES and sha256_file(Z) == DATA_SHA256):
    run(f"curl -L --fail -o {Z} {DATA_URL}", cwd=W)
size, digest = os.path.getsize(Z), sha256_file(Z)
log("size", size, "(expected", DATA_BYTES, ")  sha256", digest)
assert size == DATA_BYTES and digest == DATA_SHA256, "Dataset mismatch: stop. Do not continue with a different file."
log("Dataset verified")

07:00:30 $ curl -L --fail -o /kaggle/working/repo/data/raw/phase3-feasibility-corrected/CICIDS2017_improved.zip https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CICIDS2017_improved.zip
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0  327M    0  7962    0     0   5002      0 19:04:42  0:00:01 19:04:41  5001
  0  327M    0 1789k    0     0   691k      0  0:08:04  0:00:02  0:08:02  691k
  6  327M    6 21.9M    0     0  6205k      0  0:00:54  0:00:03  0:00:51 6205k
 13  327M   13 44.1M    0     0  9742k      0  0:00:34  0:00:04  0:00:30 9741k
 20  327M   20 66.4M    0     0  11.7M      0  0:00:27  0:00:05  0:00:22 13.5M
 25  327M   25 84.8M    0     0  12.9M      0  0:00:25  0:00:06  0:00:19 17.1M
 32  3

In [5]:
# 5. Rebuild the derived data (resumable). Order was derived from reading the tools; each step is logged.
T = "docs/phase3-plan/tools"
# Guard: an earlier background run (run_chain.sh) may still be writing the same files in this session.
if subprocess.run(["pgrep", "-f", "[r]un_chain.sh"], capture_output=True).returncode == 0:
    raise RuntimeError("The earlier background chain (run_chain.sh) is still running. Wait for it to finish (tail /kaggle/working/chain.log), then re-run this cell.")
# Adopt steps the earlier background run already completed, so they are not repeated.
old_log = f"{W}/chain.log"
if os.path.exists(old_log):
    cur = None
    for line in open(old_log, errors="ignore"):
        if line.startswith("=== START "): cur = line.split()[2]
        elif line.startswith("=== DONE") and cur and not done(cur): mark(cur); log("adopted completed step from chain.log:", cur)
if not os.path.isdir(f"{W}/docs-phase3-plan-backup"):
    shutil.copytree(f"{R}/docs/phase3-plan", f"{W}/docs-phase3-plan-backup")   # these tools overwrite report files
STEPS = [
    ("A_prepare_inputs",    [VPY, f"{T}/prepare_corrected_component_audit.py"]),
    ("B_components_base",   [VPY, f"{T}/audit_corrected_components.py"]),
    ("C_split_base",        [VPY, f"{T}/audit_candidate_split.py", "--release", "corrected"]),
    ("D_no_headers_74",     [VPY, f"{T}/audit_no_headers_candidate.py"]),
    ("E_split_reuse_proof", [VPY, f"{T}/audit_no_headers_split_reuse.py"]),
    ("F_ownership_74",      [VPY, f"{T}/audit_candidate_ownership.py", "--release", "corrected", "--profile", "no-headers-74",
                             "--benign-policy", "benign-subset-by-component"]),
]
for name, cmd in STEPS:
    if done(name): log("skip (already done):", name); continue
    log("=== START", name)
    run(cmd, logfile=f"{OUT1}/chain-{name}.log")
    mark(name); log("=== DONE", name)
log("All data-rebuild steps finished")

07:00:49 === START A_prepare_inputs
07:00:49 $ /kaggle/working/venv312/bin/python docs/phase3-plan/tools/prepare_corrected_component_audit.py
{"file": "friday.csv", "rows": 547557, "elapsed_seconds": 11.64}
{"file": "monday.csv", "rows": 371624, "elapsed_seconds": 19.67}
{"file": "thursday.csv", "rows": 362076, "elapsed_seconds": 27.13}
{"file": "tuesday.csv", "rows": 322078, "elapsed_seconds": 33.94}
{"file": "wednesday.csv", "rows": 496641, "elapsed_seconds": 44.52}
{"rows": 2099976, "elapsed_seconds": 44.524}
07:01:33 exit 0 after 45s
07:01:33 === DONE A_prepare_inputs
07:01:33 === START B_components_base
07:01:33 $ /kaggle/working/venv312/bin/python docs/phase3-plan/tools/audit_corrected_components.py
{"step": "metadata", "rows": 2099976, "invalid_metadata": 0, "pm_interpreted": 0, "elapsed_seconds": 10.26}
{"step": "pretransform_fp64_content", "valid": 2099971, "groups": 1714097, "duplicate_groups": 63664}
{"step": "frozen_T0_fp32_content", "valid": 2099971, "groups": 1713729, "du

In [6]:
# 6. Check regenerated files and reports against the committed evidence (does NOT stop; prints a verdict)
import re
ident = json.load(open(f"{R}/docs/phase3-plan/audit-local-data-identities.json"))
def crlf_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 24), b""): h.update(c.replace(bytes([10]), bytes([13, 10])))
    return h.hexdigest()
rows, bad = [], []
for f in ident["files"]:
    p = f"{R}/{f['path']}"
    if not os.path.exists(p): rows.append((f["path"], "not produced (not part of this chain)")); continue
    if sha256_file(p) == f["sha256"]: st = "MATCH"
    elif p.endswith(".csv") and crlf_sha(p) == f["sha256"]: st = "MATCH after Windows line endings (CRLF)"
    else: st = "MISMATCH"; bad.append(f["path"])
    rows.append((f["path"], st))
for path_, st in rows: print(st.ljust(42), path_)
json.dump(rows, open(f"{OUT1}/derived-hash-check.json", "w"), indent=1)

# snapshot the regenerated reports BEFORE restoring the committed (hash-pinned) ones
SNAP = f"{W}/docs-regenerated"
shutil.rmtree(SNAP, ignore_errors=True); shutil.copytree(f"{R}/docs/phase3-plan", SNAP)
changed = subprocess.run("git diff --name-only -- docs/phase3-plan", shell=True, cwd=R, capture_output=True, text=True).stdout.split()
run("git checkout -- docs/phase3-plan", cwd=R)
VOLATILE = re.compile(r"(utc|elapsed|seconds|platform|python|started|completed)", re.I)
def leaves(o, p=""):
    if isinstance(o, dict):
        for k, v in o.items():
            if not VOLATILE.search(k): yield from leaves(v, f"{p}/{k}")
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from leaves(v, f"{p}[{i}]")
    else: yield p, o
report = {}
for rel in changed:
    if not rel.endswith(".json"): continue
    new = dict(leaves(json.load(open(f"{SNAP}/{os.path.basename(rel)}"))))
    old = dict(leaves(json.load(open(f"{R}/{rel}"))))
    diffs = [(k, old.get(k), new.get(k)) for k in sorted(set(old) | set(new)) if old.get(k) != new.get(k)]
    report[rel] = diffs
    print()
    print(f"{os.path.basename(rel)}: {len(set(old)|set(new))} fields compared, {len(diffs)} differ")
    for k, a_, b_ in diffs[:12]:
        print("  ", k)
        print("      committed  =", str(a_)[:90])
        print("      regenerated=", str(b_)[:90])
json.dump({k: [[x, str(a_), str(b_)] for x, a_, b_ in v] for k, v in report.items()}, open(f"{OUT1}/report-content-diff.json", "w"), indent=1)
log("Byte-level mismatches:", len(bad), "-> send this output; the reference run (cell 7) will still refuse until the cause is settled.")

not produced (not part of this chain)      data/raw/phase3-feasibility-original/candidate-features-fp64.raw
not produced (not part of this chain)      data/raw/phase3-feasibility-original/candidate-finite.raw
not produced (not part of this chain)      data/raw/phase3-feasibility-original/feasibility-metadata.csv
not produced (not part of this chain)      data/raw/phase3-feasibility-original/conditional-components-120.npz
not produced (not part of this chain)      data/raw/phase3-feasibility-original/conditional-candidate-split.npz
not produced (not part of this chain)      data/raw/phase3-feasibility-original/domain-checked/conditional-components-120.npz
MATCH                                      data/raw/phase3-feasibility-corrected/candidate-features-fp64.raw
MATCH                                      data/raw/phase3-feasibility-corrected/candidate-finite.raw
MATCH after Windows line endings (CRLF)    data/raw/phase3-feasibility-corrected/component-metadata.csv
MISMATCH              

In [7]:
# 6b. Reproduce Ahmed's exact file bytes (Windows line endings / zip host-OS byte). A file is replaced ONLY if the result
# equals its pinned SHA-256, so no pin is weakened and prepare.py stays untouched.
import zipfile, struct
pins = {f["path"]: f["sha256"] for f in json.load(open(f"{R}/docs/phase3-plan/audit-local-data-identities.json"))["files"]}
def to_crlf(path, out):
    with open(path, "rb") as f, open(out, "wb") as g:
        for c in iter(lambda: f.read(1 << 24), b""): g.write(c.replace(bytes([10]), bytes([13, 10])))
def retag_npz(path, os_byte, ext):
    z = zipfile.ZipFile(path); pos = z.start_dir; n = len(z.infolist()); z.close()
    data = bytearray(open(path, "rb").read())
    for _ in range(n):
        assert data[pos:pos+4] == b"PK"
        data[pos+5] = os_byte
        if ext is not None: data[pos+38:pos+42] = struct.pack("<I", ext)
        nl, el, cl = struct.unpack("<HHH", data[pos+28:pos+34]); pos += 46 + nl + el + cl
    return bytes(data)
res = []
for rel, want in pins.items():
    p = f"{R}/{rel}"
    if not os.path.exists(p) or sha256_file(p) == want: continue
    if rel.endswith(".csv"):
        tmp = p + ".tmp"; to_crlf(p, tmp)
        if sha256_file(tmp) == want: os.replace(tmp, p); res.append((rel, "FIXED: CRLF line endings"))
        else: os.remove(tmp); res.append((rel, "still different"))
    elif rel.endswith(".npz"):
        hit = None
        for os_byte, ext in [(0, None), (0, 0), (0, 0o600 << 16)]:
            b = retag_npz(p, os_byte, ext)
            if hashlib.sha256(b).hexdigest() == want: hit = (os_byte, ext); break
        if hit: open(p + ".tmp", "wb").write(b); os.replace(p + ".tmp", p); res.append((rel, f"FIXED: zip host-OS byte -> {hit[0]}"))
        else: res.append((rel, "still different"))
for rel, st in res: print(st.ljust(40), rel.split("feasibility-")[1])
json.dump(res, open(f"{OUT1}/byte-normalisation.json", "w"), indent=1)

FIXED: CRLF line endings                 corrected/component-metadata.csv
still different                          corrected/conditional-components-120.npz
FIXED: zip host-OS byte -> 0             corrected/conditional-candidate-split.npz
FIXED: zip host-OS byte -> 0             corrected/no-headers-74/conditional-components-120.npz
FIXED: zip host-OS byte -> 0             corrected/no-headers-74/conditional-candidate-split.npz
FIXED: zip host-OS byte -> 0             corrected/no-headers-74/conditional-ownership-benign-subset-seed-1.npz
FIXED: zip host-OS byte -> 0             corrected/no-headers-74/conditional-ownership-benign-subset-seed-10.npz
FIXED: zip host-OS byte -> 0             corrected/no-headers-74/conditional-ownership-benign-subset-seed-11.npz
FIXED: zip host-OS byte -> 0             corrected/no-headers-74/conditional-ownership-benign-subset-seed-2.npz
FIXED: zip host-OS byte -> 0             corrected/no-headers-74/conditional-ownership-benign-subset-seed-3.npz
FIXED:

In [8]:
# A-8. Reference data package (the audited 74-feature package) built with Windows-equivalent zip headers, then every pinned file is verified
P = "data/processed/phase3-reference74-v1"
WRAP = f"{W}/win_zip_wrapper.py"
open(WRAP, "w").write(chr(10).join(["import sys, os, zipfile, runpy", "sys.path.insert(0, os.getcwd())", "_o = zipfile.ZipInfo.__init__", "def _i(self, *a, **k):", "    _o(self, *a, **k); self.create_system = 0", "zipfile.ZipInfo.__init__ = _i", "m = sys.argv[1]; sys.argv = [m] + sys.argv[2:]", "runpy.run_module(m, run_name='__main__', alter_sys=True)", ""]))
if not done("ref_prepare"):
    run([VPY, WRAP, "src.phase3.prepare", "--output", P], logfile=f"{OUT1}/ref-prepare.log"); mark("ref_prepare")
pins = json.load(open(f"{R}/docs/phase3-reference/evidence/preparation-rebuild.json"))["files"]
bad = [n for n, want in pins.items() if not os.path.exists(f"{R}/{P}/{n}") or sha256_file(f"{R}/{P}/{n}") != want]
log("pinned preparation files:", len(pins), "| not matching:", bad)
assert not bad, "The rebuilt package differs from the pinned hashes: stop and send this output."
log("All pinned preparation files match.")

07:08:40 $ /kaggle/working/venv312/bin/python /kaggle/working/win_zip_wrapper.py src.phase3.prepare --output data/processed/phase3-reference74-v1
{"cross_role_component_overlap":0,"development_manifest_sha256":"2d8e209d17777b8ac5b2cc7f32dfb90fd5c358a671b79dcc3715b8368e57f081","development_rows":272506,"ownership_objective":0.6401340441348633,"ownership_sha256":"2c237b359adfaede9efc80258d6e632e3a2d82058df04ea87eca39581704d475","protocol_sha256":"a8fc9dc310716d104a1df24f4eef24885af46481d8c6d6c25c6d2a302388127a","reconstruction_scope":"Hash-checked audited graph/split; independently rebuilt benign pool/seed-101 ownership/T0 transform on all exported records.","scaler_sha256":"498d267639684c44fa771229c18b61ed92b012ccff8d80e7e45bf9352910b6cd","scientific_identity":"d88dbe33b9fab108061d1bfd2048f1a224086c442a12eeb3e0194278c8f05ad6","source_assets":{"data/raw/phase3-feasibility-corrected/component-metadata.csv":"e079f704b7ae0c8ca42f230e431527632852ca4efa3469970d71825abb6aaf1f","data/raw/phase3

In [9]:
# A-9. One training package per development seed (clean supplied labels; poisoning is applied later, inside the experiment, with nested doses)
SEEDS_ALL = [101, 102, 103]            # the only development seeds the repository allows (104 and above are sealed for the final evaluation)
PKG = {}
for seed in SEEDS_ALL:
    out = f"data/processed/own/s{seed}"
    if not os.path.exists(f"{R}/{out}/preparation.json"):
        run([VPY, "-m", "src.phase3_v2.prepare", "--output", out, "--seed", str(seed), "--dose", "0", "1"], logfile=f"{OUT1}/prepare-s{seed}.log")
    prep = json.load(open(f"{R}/{out}/preparation.json"))
    PKG[seed] = f"{R}/{out}/train"
    log(f"seed {seed}: ready | training rows {prep['training_rows']} | manifest {prep['train_manifest_sha256'][:16]}... | test rows exported {prep['test_rows_exported']}")
    assert prep["test_rows_exported"] == 0 and prep["development_feature_rows_in_train"] == 0
DEV_DIR = f"{R}/{P}/development"
TAX_PATH = f"{R}/configs/phase3/taxonomy-v1.json"
assert os.path.exists(f"{DEV_DIR}/x.npy") and os.path.exists(TAX_PATH)
log("development bank and taxonomy found")

07:09:05 $ /kaggle/working/venv312/bin/python -m src.phase3_v2.prepare --output data/processed/own/s101 --seed 101 --dose 0 1
16255e585081177b79ca7552ff8b3793d1c04918f82a3352c871417734e3b5cd
07:09:16 exit 0 after 11s
07:09:16 seed 101: ready | training rows 183045 | manifest 16255e585081177b... | test rows exported 0
07:09:16 $ /kaggle/working/venv312/bin/python -m src.phase3_v2.prepare --output data/processed/own/s102 --seed 102 --dose 0 1
717749d4137010eb4c2147921caa54870e6eec3a898fee5f056efc9eb704eed9
07:09:27 exit 0 after 11s
07:09:27 seed 102: ready | training rows 183045 | manifest 717749d4137010eb... | test rows exported 0
07:09:27 $ /kaggle/working/venv312/bin/python -m src.phase3_v2.prepare --output data/processed/own/s103 --seed 103 --dose 0 1
d9b2cbb537f2d3b2e4c8640073cc0a48c6ec2e16fd02f8d83a3420432117e5d1
07:09:38 exit 0 after 11s
07:09:38 seed 103: ready | training rows 183045 | manifest d9b2cbb537f2d3b2... | test rows exported 0
07:09:38 development bank and taxonomy foun

In [10]:
# A-10. Dataset and preprocessing report (Phase 3 requirement 2) and PART A package
import numpy as np, zipfile
tax = json.load(open(TAX_PATH)); cls = sorted(tax["classes"], key=lambda c: c["source_code"])
rows = []
for c in cls:
    k = c["source_code"]
    tr = [int((np.load(f"{PKG[101]}/E{s}/y.npy", mmap_mode="r") == k).sum()) for s in range(1, 5)]
    dv = int((np.load(f"{DEV_DIR}/y.npy", mmap_mode="r") == k).sum())
    rows.append([k, c["label"], c["arrival_stage"], c["scoring_eligible"], sum(tr), dv])
import csv
with open(f"{OUT1}/dataset-classes.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["code", "class", "arrival_stage", "scoring_eligible", "train_rows_seed101", "dev_rows"]); w.writerows(rows)
for r in rows: print(str(r[0]).rjust(2), r[1][:28].ljust(28), "stage", r[2], "elig" if r[3] else "    ", "train", str(r[4]).rjust(7), "dev", str(r[5]).rjust(7))
open(f"{OUT1}/preprocessing.md", "w").write("""# Dataset and preprocessing
- Source: CICIDS2017-improved (corrected labels), downloaded and verified by SHA-256.
- 74 header-free flow features, repository-audited split; every pinned preparation file verified (A-8).
- Four arrival stages E1-E4 with the taxonomy in configs/phase3/taxonomy-v1.json; the class table is in dataset-classes.csv.
- Five clients: whole connected components are owned by exactly one client (no row-level leakage between clients); ownership is rebuilt per seed (101, 102, 103).
- Development bank only. Test rows exported: 0. Development rows in training packages: 0.
""")
with zipfile.ZipFile(f"{W}/phase3-part1-outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT1):
        for fn in files: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), W))
    for seed in SEEDS_ALL: z.write(f"{R}/data/processed/own/s{seed}/preparation.json", f"outputs/part1/preparation-s{seed}.json")
mark("part1_complete"); log("PART A COMPLETE -> /kaggle/working/phase3-part1-outputs.zip")

 0 Benign                       stage 1 elig train   14000 dev  216158
 1 Bot                          stage 4 elig train     441 dev     148
 2 DDoS                         stage 4 elig train   57086 dev   19029
 3 DoS GoldenEye                stage 2 elig train    4540 dev    1513
 4 DoS Hulk                     stage 2 elig train   95080 dev   31694
 5 DoS Slowhttptest             stage 2 elig train    1037 dev     346
 6 DoS Slowloris                stage 2 elig train    2071 dev     691
 7 FTP-Patator                  stage 1 elig train    2382 dev     795
 8 Heartbleed                   stage 2      train      11 dev       0
 9 Infiltration                 stage 3      train      27 dev       5
10 Infiltration - Portscan      stage 3 elig train    3591 dev    1198
11 PortScan                     stage 4 elig train     943 dev     315
12 SSH-Patator                  stage 1 elig train    1776 dev     592
13 Web Attack Brute Force       stage 3      train      43 dev      15
14 Web

# PART B: implementation, baselines and ablations (parallel, CPU or GPU)
Own implementation (`p3lib.py`): federated continual learning with five clients, growing head, learners **FT / EI / DR**, label poisoning (Hulk -> benign on one client in E2, nested doses), admission rules **DC / DM / DQ**, full metrics. `p3parallel.py` runs many experiments at once in separate processes, one result file per run.

In [11]:
# B-1. Library files (written to disk so that worker processes can import them)
import os
CODE = f"{W}/code"; os.makedirs(CODE, exist_ok=True)
LIB_SRC = """\"\"\"Phase 3: own implementation of a federated continual intrusion-detection study with label poisoning.

Five simulated clients learn four stages (E1-E4) of attack families with a growing classification head.
Learners: FT (fine-tuning), EI (EWC-style, per-example diagonal Fisher), DR (DER++ replay).
Admission rules at each stage: NONE, DC (class-aware quota), DM (matched-volume global ranking), DQ (class-aware quota, random choice).
Poisoning: one client relabels a fraction of its DoS-Hulk training rows as benign during E2 only (withdrawn in E3-E4).
Everything is evaluated on a fixed development bank; no final-test data is read here.
Runs execute on CPU or GPU ('device'); all data and the replay memory live on that device. CPU and GPU results are not bit-identical,
so every paired comparison must use runs from one device type.
\"\"\"
from __future__ import annotations

import json
import math
import time
from dataclasses import asdict, dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call, grad, vmap

D_IN, CMAX, N_CLIENTS = 74, 16, 5
BENIGN, HULK = 0, 4


# ----------------------------------------------------------------------------- configuration and data
@dataclass(frozen=True)
class Cfg:
    learner: str = "FT"          # FT | EI | DR
    admission: str = "NONE"      # NONE | DC | DM | DQ  (DR only)
    seed: int = 101
    dose: float = 0.0            # fraction of each attacking client's Hulk rows relabelled benign in E2
    n_att: int = 1               # number of attacking clients (nested: the first k of a seeded order)
    rep: int = 0                 # training replicate within one ownership partition (0 = the original run)
    rounds: int = 12             # synchronisation rounds per stage
    epochs: int = 1              # local epochs per round
    lr: float = 1e-3
    batch: int = 256
    lam_E: float = 100.0         # EI penalty weight
    lam_y: float = 1.0           # DR replay-label weight
    lam_z: float = 0.1           # DR logit-matching weight
    mem: int = 500               # DR memory records per client
    replay_batch: int = 128
    buffer: str = "reservoir"    # reservoir | fifo (ablation)
    q: float = 0.75              # keep fraction of the class-aware quota
    nmin: int = 20               # classes smaller than this are kept whole
    fisher_n: int = 1024
    dropout: float = 0.1

    @property
    def sd(self) -> int:
        return self.seed + 1000 * self.rep        # seed for every random stream; the data package is chosen by self.seed only

    def tag(self) -> str:
        s = f"{self.learner}-{self.admission}-d{int(round(self.dose * 100)):03d}-k{self.n_att}-r{self.rep}-s{self.seed}"
        extras = []
        if self.learner == "DR":
            if self.lam_z != 0.1: extras.append(f"z{self.lam_z:g}")
            if self.lam_y != 1.0: extras.append(f"y{self.lam_y:g}")
            if self.buffer != "reservoir": extras.append(self.buffer)
        for name, default in (("mem", 500), ("q", 0.75), ("nmin", 20), ("lr", 1e-3), ("lam_E", 100.0), ("rounds", 12)):
            if getattr(self, name) != default: extras.append(f"{name}{getattr(self, name):g}")
        return s + ("-" + "-".join(extras) if extras else "")


def load_taxonomy(path):
    t = json.load(open(path))
    cls = sorted(t["classes"], key=lambda c: c["source_code"])
    return {"names": {c["source_code"]: c["label"] for c in cls},
            "arrival": {c["source_code"]: c["arrival_stage"] for c in cls},
            "eligible": {c["source_code"] for c in cls if c["scoring_eligible"]},
            "dev_rows": {c["source_code"]: c["role_support_before_benign_selection"]["development"]["rows"] for c in cls},
            "train_rows": {c["source_code"]: c["selected_training_rows"] for c in cls}}


def load_stages(train_dir):
    out = []
    for s in range(1, 5):
        d = {k: np.load(f"{train_dir}/E{s}/{k}.npy") for k in ("x", "y", "ids", "owner")}
        out.append({"x": d["x"].astype(np.float32), "y": d["y"].astype(np.int64), "ids": d["ids"], "owner": d["owner"].astype(np.int64)})
    return out


def load_dev(dev_dir):
    return {"x": np.load(f"{dev_dir}/x.npy").astype(np.float32), "y": np.load(f"{dev_dir}/y.npy").astype(np.int64)}


def check_data(stages, dev, tax):
    \"\"\"Integrity checks against the adopted taxonomy; returns (ok, messages).\"\"\"
    msgs, ok = [], True
    for c, n in tax["dev_rows"].items():
        got = int((dev["y"] == c).sum())
        if got != n: ok = False; msgs.append(f"dev class {c} ({tax['names'][c]}): {got} rows, taxonomy says {n}")
    for c, n in tax["train_rows"].items():
        got = sum(int((s["y"] == c).sum()) for s in stages)
        if got != n: ok = False; msgs.append(f"train class {c} ({tax['names'][c]}): {got} rows, taxonomy says {n}")
    for t, s in enumerate(stages, 1):
        arrived = {int(c) for c in np.unique(s["y"])} - {BENIGN}
        wrong = {c for c in arrived if tax["arrival"][c] != t}
        if wrong: ok = False; msgs.append(f"stage {t}: classes {sorted(wrong)} arrive in another stage per taxonomy")
    return ok, msgs


_UPLOADS = {}                    # device copies of large read-only arrays, kept for the life of the process


def upload(arr, device):
    key = (id(arr), str(device))
    if key not in _UPLOADS: _UPLOADS[key] = (arr, torch.from_numpy(arr).to(device))   # keep arr alive so id() stays unique
    return _UPLOADS[key][1]


# ----------------------------------------------------------------------------- model
class Net(nn.Module):
    def __init__(self, width, hidden=(128, 64), p=0.1):
        super().__init__()
        self.f1, self.f2, self.head, self.p = nn.Linear(D_IN, hidden[0]), nn.Linear(hidden[0], hidden[1]), nn.Linear(hidden[1], width), p

    def forward(self, x):
        h = F.dropout(F.relu(self.f1(x)), self.p, self.training)
        h = F.dropout(F.relu(self.f2(h)), self.p, self.training)
        return self.head(h)

    def grow(self, width, gen):
        old = self.head
        if width == old.out_features: return
        new = nn.Linear(old.in_features, width).to(old.weight.device)
        with torch.no_grad():
            new.weight.copy_((torch.randn(new.weight.shape, generator=gen) * 0.01).to(old.weight.device))
            new.bias.zero_()
            new.weight[: old.out_features] = old.weight
            new.bias[: old.out_features] = old.bias
        self.head = new


def n_params(model): return sum(p.numel() for p in model.parameters())


def clone_net(model):
    m = Net(model.head.out_features, (model.f1.out_features, model.f2.out_features), model.p).to(model.head.weight.device)
    m.load_state_dict(model.state_dict())
    return m


def fedavg(states, weights):
    \"\"\"Sample-count weighted average, computed on the CPU in float64 for a device-independent reduction order.\"\"\"
    w = np.asarray(weights, dtype=np.float64); w = w / w.sum()
    return {k: sum(float(wi) * s[k].detach().cpu().double() for wi, s in zip(w, states)).float() for k in states[0]}


# ----------------------------------------------------------------------------- poisoning and admission
def poison_stage(stage, cfg):
    \"\"\"Relabel a prefix of each attacking client's (random, seeded) Hulk ordering as benign.
    Doses are nested prefixes; attackers are nested too (the first k of a seeded client order), so k=1 is contained in k=3 and k=5.\"\"\"
    y = stage["y"].copy(); mask = np.zeros(len(y), bool)
    info = {"attacked_client": None, "attacked_clients": [], "eligible_clients": [], "eligible_hulk_rows": 0,
            "hulk_rows_stage": int((y == HULK).sum()), "flipped": 0}
    if cfg.dose <= 0: return y, mask, info
    counts = np.bincount(stage["owner"][y == HULK], minlength=N_CLIENTS)
    eligible = np.flatnonzero(counts >= 10)
    if not len(eligible): raise ValueError("No client holds ten Hulk rows")
    first = int(np.random.default_rng([cfg.sd, 2]).choice(eligible))
    rest = [int(c) for c in eligible if c != first]
    if rest: rest = [int(c) for c in np.random.default_rng([cfg.sd, 4]).permutation(rest)]
    attackers = ([first] + rest)[: min(max(cfg.n_att, 1), len(eligible))]
    total_src = flipped = 0
    for attacked in attackers:
        src = np.flatnonzero((stage["owner"] == attacked) & (y == HULK))
        src = src[np.argsort(stage["ids"][src], kind="stable")]
        perm = np.random.default_rng([cfg.sd, 3, attacked]).permutation(src)
        q = int(math.floor(cfg.dose * len(src)))
        y[perm[:q]] = BENIGN; mask[perm[:q]] = True
        total_src += len(src); flipped += q
    info.update(attacked_client=attackers[0], attacked_clients=attackers, eligible_clients=eligible.tolist(), eligible_hulk_rows=int(total_src), flipped=int(flipped))
    return y, mask, info


@torch.no_grad()
def row_losses(model, x_t, ys_t, chunk=16384):
    model.eval(); out = []
    for i in range(0, len(x_t), chunk):
        out.append(F.cross_entropy(model(x_t[i:i + chunk]), ys_t[i:i + chunk], reduction="none"))
    return torch.cat(out).cpu().numpy()


def admit(cfg, model, x_t, ys_t, ys, owner, ids, new_slots, rng):
    \"\"\"Per-client admitted row indices (sorted). Scores come from the frozen model after head expansion.\"\"\"
    per_client = [np.flatnonzero(owner == k) for k in range(N_CLIENTS)]
    if cfg.admission == "NONE": return per_client
    loss = row_losses(model, x_t, ys_t)
    chosen = []
    for rows in per_client:
        if len(rows) == 0: chosen.append(rows); continue
        yk = ys[rows]; quota = {}
        for c in np.unique(yk):
            n = int((yk == c).sum())
            quota[int(c)] = n if (int(c) in new_slots or n < cfg.nmin) else max(1, math.ceil(cfg.q * n))
        total = sum(quota.values())
        if cfg.admission == "DM":                                        # same volume, global ranking (loss ascending, ties by id)
            pick = rows[np.lexsort((ids[rows], loss[rows]))][:total]
        else:                                                            # DC: lowest loss within class; DQ: random within class
            pick = []
            for c, k in quota.items():
                cls_rows = rows[yk == c]
                if cfg.admission == "DC": pick.append(cls_rows[np.lexsort((ids[cls_rows], loss[cls_rows]))][:k])
                else: pick.append(cls_rows[rng.permutation(len(cls_rows))[:k]])
            pick = np.concatenate(pick)
        chosen.append(np.sort(pick))
    return chosen


# ----------------------------------------------------------------------------- DR memory (device resident)
class Buffer:
    def __init__(self, m, kind, rng, device):
        self.m, self.kind, self.rng, self.device, self.size, self.seen, self.ptr = m, kind, rng, device, 0, 0, 0
        self.x = torch.zeros((m, D_IN), device=device); self.y = torch.zeros(m, dtype=torch.long, device=device)
        self.z = torch.zeros((m, CMAX), device=device); self.w = torch.ones(m, dtype=torch.long, device=device)
        self.poison = np.zeros(m, bool)

    def insert(self, xb, yb, zb, w, poison):
        \"\"\"Sequential reservoir (Algorithm R) or FIFO semantics; one device scatter per batch (last writer wins).\"\"\"
        last = {}
        for i in range(len(poison)):
            self.seen += 1
            if self.size < self.m: j = self.size; self.size += 1
            elif self.kind == "fifo": j = self.ptr; self.ptr = (self.ptr + 1) % self.m
            else:
                j = int(self.rng.integers(1, self.seen + 1))
                if j > self.m: continue
                j -= 1
            last[j] = i
        if not last: return
        jj = np.fromiter(last.keys(), np.int64, len(last)); ii = np.fromiter(last.values(), np.int64, len(last))
        jt, it = torch.from_numpy(jj).to(self.device), torch.from_numpy(ii).to(self.device)
        self.x[jt] = xb[it]; self.y[jt] = yb[it]; self.w[jt] = w
        self.z[jt] = F.pad(zb, (0, CMAX - zb.shape[1]))[it]
        self.poison[jj] = poison[ii]

    def sample(self, k):
        k = min(k, self.size); idx = torch.from_numpy(self.rng.choice(self.size, k, replace=False)).to(self.device)
        return self.x[idx], self.y[idx], self.z[idx], self.w[idx]

    def bytes(self):
        return int(self.size * (D_IN * 4 + CMAX * 4 + 8 + 8))


# ----------------------------------------------------------------------------- EWC-style importance
def fisher_diag(model, x_t, y_t):
    model.eval()
    params = {k: v.detach() for k, v in model.named_parameters()}

    def loss(p, xi, yi): return F.cross_entropy(functional_call(model, p, (xi.unsqueeze(0),)), yi.unsqueeze(0))
    g = vmap(grad(loss), in_dims=(None, 0, 0))(params, x_t, y_t)
    return {k: (v ** 2).mean(0) for k, v in g.items()}


def pad_like(t, shape):
    if t.shape == shape: return t
    out = torch.zeros(shape, dtype=t.dtype, device=t.device); out[tuple(slice(0, s) for s in t.shape)] = t; return out


# ----------------------------------------------------------------------------- evaluation
@torch.no_grad()
def predict_codes(model, slots, x_t, chunk=32768):
    model.eval(); slots = np.asarray(slots); out = []
    for i in range(0, len(x_t), chunk): out.append(model(x_t[i:i + chunk]).argmax(1).cpu().numpy())
    return slots[np.concatenate(out)]


def stage_metrics(model, slots, dev, dev_x_t, tax, t, device):
    seen = [c for c, a in tax["arrival"].items() if a <= t]
    idx = np.flatnonzero(np.isin(dev["y"], seen)); yt = dev["y"][idx]
    yp = predict_codes(model, slots, dev_x_t[torch.from_numpy(idx).to(device)])
    conf = np.zeros((CMAX, CMAX), np.int64); np.add.at(conf, (yt, yp), 1)
    rows = {}
    for c in seen:
        sup = int(conf[c].sum())
        if sup == 0: continue
        tp, fn, fp = int(conf[c, c]), sup - int(conf[c, c]), int(conf[:, c].sum() - conf[c, c])
        prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / sup
        rows[c] = {"support": sup, "tp": tp, "fn": fn, "fp": fp, "precision": prec, "recall": rec,
                   "f1": (2 * prec * rec / (prec + rec)) if prec + rec else 0.0}
    elig = [c for c in rows if c in tax["eligible"]]
    benign_n = int(conf[BENIGN].sum())
    out = {"stage": t, "slots": [int(s) for s in slots], "classes": {int(c): r for c, r in rows.items()}, "eligible": [int(c) for c in elig],
           "macro_f1": float(np.mean([rows[c]["f1"] for c in elig])), "accuracy": float(np.trace(conf) / conf.sum()),
           "benign_far": float(1.0 - conf[BENIGN, BENIGN] / benign_n), "benign_rows": benign_n, "confusion": conf.tolist()}
    if HULK in rows:
        out["hulk_recall"] = rows[HULK]["recall"]; out["hulk_asr"] = float(conf[HULK, BENIGN] / conf[HULK].sum())
    return out


def summarize(stage_results, tax):
    \"\"\"Acquisition A (untargeted new eligible classes, Hulk excluded), cohort recall matrix, BWT and forgetting.\"\"\"
    A_vals = []
    for r in stage_results:
        t = r["stage"]
        new = [c for c in r["eligible"] if tax["arrival"][c] == t and c not in (BENIGN, HULK) and t >= 2]
        if new: A_vals.append(float(np.mean([r["classes"][c]["f1"] for c in new])))
    cohorts = {j: [c for c in stage_results[-1]["eligible"] if tax["arrival"][c] == j and c != BENIGN] for j in range(1, 5)}
    R = {}
    for r in stage_results:
        for j, cs in cohorts.items():
            if cs and j <= r["stage"]: R[(r["stage"], j)] = float(np.mean([r["classes"][c]["recall"] for c in cs if c in r["classes"]]))
    js = [j for j in range(1, 4) if (4, j) in R and (j, j) in R]
    bwt = float(np.mean([R[(4, j)] - R[(j, j)] for j in js])) if js else float("nan")
    forg = float(np.mean([max(R[(t, j)] for t in range(j, 5) if (t, j) in R) - R[(4, j)] for j in js])) if js else float("nan")
    return {"A": float(np.mean(A_vals)) if A_vals else float("nan"), "BWT": bwt, "forgetting": forg,
            "cohort_recall": {f"{t}-{j}": v for (t, j), v in R.items()}}


# ----------------------------------------------------------------------------- the experiment
def run(cfg: Cfg, stages, dev, tax, threads=1, device="cpu"):
    if cfg.learner not in ("FT", "EI", "DR"): raise ValueError("learner")
    if cfg.admission not in ("NONE", "DC", "DM", "DQ"): raise ValueError("admission")
    if cfg.admission != "NONE" and cfg.learner != "DR": raise ValueError("admission controls use DR")
    device = torch.device(device)
    torch.set_num_threads(threads); torch.manual_seed(cfg.sd)
    t0 = time.time(); gen = torch.Generator().manual_seed(cfg.sd)
    rng_adm = np.random.default_rng([cfg.sd, 5])
    dev_x_t = upload(dev["x"], device)
    model, slots, anchor, imp = None, [], None, None
    buffers = [Buffer(cfg.mem, cfg.buffer, np.random.default_rng([cfg.sd, 7, k]), device) for k in range(N_CLIENTS)] if cfg.learner == "DR" else None
    results, diag, comm_bytes, poison_info = [], [], 0, {}
    for t in range(1, 5):
        s = stages[t - 1]; ids, owner = s["ids"], s["owner"]; x_t = upload(s["x"], device)
        if t == 2: y, pmask, poison_info = poison_stage(s, cfg)
        else: y, pmask = s["y"], np.zeros(len(s["y"]), bool)
        new_codes = sorted(set(np.unique(y).tolist()) - set(slots)); slots = slots + new_codes
        code2slot = np.full(CMAX, -1, np.int64); code2slot[slots] = np.arange(len(slots)); ys = code2slot[y]
        ys_t = torch.from_numpy(ys).to(device)
        if model is None: model = Net(len(slots), p=cfg.dropout).to(device)
        else:
            model.grow(len(slots), gen)
            if anchor is not None:                                  # new head rows: anchor = fresh initial value, importance = 0
                cur = dict(model.named_parameters())
                for n in anchor:
                    if anchor[n].shape != cur[n].shape:
                        a = cur[n].detach().clone(); a[tuple(slice(0, sz) for sz in anchor[n].shape)] = anchor[n]
                        anchor[n] = a; imp[n] = pad_like(imp[n], cur[n].shape)
        new_slots = {int(code2slot[c]) for c in new_codes}
        admitted = admit(cfg, model, x_t, ys_t, ys, owner, ids, new_slots, rng_adm)
        n_adm = [len(a) for a in admitted]
        d = {"stage": t, "rows": int(len(ys)), "admitted": int(sum(n_adm)), "poisoned_rows": int(pmask.sum()),
             "admitted_poisoned": int(sum(int(pmask[a].sum()) for a in admitted)), "per_client_admitted": n_adm,
             "new_class_rows": int(np.isin(y, new_codes).sum()), "new_class_rows_admitted": int(sum(int(np.isin(y[a], new_codes).sum()) for a in admitted))}
        active = [k for k in range(N_CLIENTS) if n_adm[k] > 0]
        for r in range(cfg.rounds):
            states = []
            for k in active:
                local = clone_net(model); opt = torch.optim.Adam(local.parameters(), lr=cfg.lr)
                rows = admitted[k]; lrng = np.random.default_rng([cfg.sd, t, r, k]); torch.manual_seed(cfg.sd * 1000 + t * 100 + r * 10 + k)
                for ep in range(cfg.epochs):
                    perm = rows[lrng.permutation(len(rows))]; perm_t = torch.from_numpy(perm).to(device)
                    for i in range(0, len(perm), cfg.batch):
                        bt = perm_t[i:i + cfg.batch]; xb, yb = x_t[bt], ys_t[bt]
                        local.train(); loss = F.cross_entropy(local(xb), yb)
                        if cfg.learner == "EI" and anchor is not None:
                            pen = sum((imp[n] * (p - anchor[n]) ** 2).sum() for n, p in local.named_parameters())
                            loss = loss + 0.5 * cfg.lam_E * pen
                        first = (r == 0 and ep == 0)
                        z_new = None
                        if cfg.learner == "DR":
                            if first:
                                local.eval()
                                with torch.no_grad(): z_new = local(xb)                                  # pre-update teacher logits
                                local.train()
                            buf = buffers[k]
                            if buf.size > 0:
                                xr, yr, zr, wr = buf.sample(cfg.replay_batch)
                                if cfg.lam_y > 0: loss = loss + cfg.lam_y * F.cross_entropy(local(xr), yr)
                                if cfg.lam_z > 0:
                                    local.eval(); outr = local(xr); local.train()
                                    pad = F.pad(outr, (0, CMAX - outr.shape[1])); mask = (torch.arange(CMAX, device=device)[None, :] < wr[:, None]).float()
                                    loss = loss + cfg.lam_z * ((((pad - zr) * mask) ** 2).sum(1) / wr.float()).mean()
                        opt.zero_grad(); loss.backward(); opt.step()
                        if z_new is not None: buffers[k].insert(xb, yb, z_new, len(slots), pmask[perm[i:i + cfg.batch]])
                states.append(local.state_dict())
            model.load_state_dict(fedavg(states, [n_adm[k] for k in active]))
            comm_bytes += 2 * 4 * n_params(model) * len(active)
        if cfg.learner == "EI":                                      # client-local Fisher at the final shared model, weighted average, accumulated
            fs, ws = [], []
            for k in active:
                rows = admitted[k]; sel = rows[np.random.default_rng([cfg.sd, 11, t, k]).permutation(len(rows))[: cfg.fisher_n]]
                st = torch.from_numpy(sel).to(device); fs.append(fisher_diag(model, x_t[st], ys_t[st])); ws.append(n_adm[k])
            w = np.asarray(ws, float) / sum(ws); new = {n: sum(float(wi) * f[n] for wi, f in zip(w, fs)) for n in fs[0]}
            imp = new if imp is None else {n: imp[n] + new[n] for n in new}
            anchor = {n: p.detach().clone() for n, p in model.named_parameters()}
        m = stage_metrics(model, slots, dev, dev_x_t, tax, t, device); results.append(m); diag.append(d)
    if device.type == "cuda": torch.cuda.synchronize()
    summ = summarize(results, tax)
    return {"cfg": asdict(cfg), "tag": cfg.tag(), "device": str(device), "stages": results, "admission": diag, "poison": poison_info, "summary": summ,
            "cost": {"seconds": time.time() - t0, "model_bytes": 4 * n_params(model), "comm_bytes_total": int(comm_bytes),
                     "buffer_bytes_total": int(sum(b.bytes() for b in buffers)) if buffers else 0,
                     "buffer_poisoned_records_end": int(sum(int(b.poison[: b.size].sum()) for b in buffers)) if buffers else 0}}
"""
PAR_SRC = """\"\"\"Parallel experiment runner for p3lib.

Each worker is a separate process (spawn) that loads the data once, keeps it on its device, and runs one experiment at a time.
GPU mode: workers share the visible GPU(s) round-robin. Every run is written to its own JSON file, so a stopped session resumes
by skipping finished runs. Results of different devices are never mixed inside one comparison (the device is stored in every file).
\"\"\"
from __future__ import annotations

import json
import multiprocessing as mp
import os
import time
import traceback
from dataclasses import asdict

_G = {}


def _init(ctx):
    import torch
    import p3lib as L
    idx = (mp.current_process()._identity or (1,))[0] - 1
    device = "cpu"
    if ctx["device"] == "cuda":
        if not torch.cuda.is_available(): raise RuntimeError("device='cuda' requested but CUDA is not available")
        device = f"cuda:{idx % torch.cuda.device_count()}"
    torch.set_num_threads(ctx["threads"])
    _G.update(ctx=ctx, device=device, L=L, tax=L.load_taxonomy(ctx["tax_path"]), dev=L.load_dev(ctx["dev_dir"]), stages={})


def _task(job):
    cfg_dict, out_path = job
    L, ctx = _G["L"], _G["ctx"]
    t0 = time.time()
    try:
        cfg = L.Cfg(**cfg_dict)
        if cfg.seed not in _G["stages"]:                      # keep one seed in memory at a time (12 seeds would not fit otherwise)
            _G["stages"].clear(); L._UPLOADS.clear()
            _G["stages"][cfg.seed] = L.load_stages(ctx["pkg_dirs"][str(cfg.seed)])
        r = L.run(cfg, _G["stages"][cfg.seed], _G["dev"], _G["tax"], threads=ctx["threads"], device=_G["device"])
        tmp = out_path + ".tmp"
        with open(tmp, "w") as f: json.dump(r, f)
        os.replace(tmp, out_path)
        return ("ok", os.path.basename(out_path), time.time() - t0, _G["device"])
    except Exception:
        return ("error", os.path.basename(out_path), traceback.format_exc(), _G["device"])


def run_grid(cfgs, out_dir, ctx, workers=4):
    \"\"\"ctx: dict(device='cpu'|'cuda', threads=int, tax_path=..., dev_dir=..., pkg_dirs={'101': train_dir, ...}).\"\"\"
    os.makedirs(out_dir, exist_ok=True)
    tags = [c.tag() for c in cfgs]
    assert len(set(tags)) == len(tags), "duplicate experiment tags in the grid"
    jobs = [(asdict(c), os.path.join(out_dir, f"{c.tag()}.json")) for c in cfgs]
    todo = sorted((j for j in jobs if not os.path.exists(j[1])), key=lambda j: j[0]["seed"])   # same-seed jobs run together
    print(f"{len(jobs)} runs in the grid, {len(jobs) - len(todo)} already finished, {len(todo)} to run on {workers} worker(s) [{ctx['device']}]", flush=True)
    if not todo: return {"ok": 0, "error": 0, "skipped": len(jobs)}
    t0, ok, errors, times = time.time(), 0, [], []
    with mp.get_context("spawn").Pool(min(workers, len(todo)), initializer=_init, initargs=(ctx,)) as pool:
        for status, name, info, dev in pool.imap_unordered(_task, todo):
            if status == "ok":
                ok += 1; times.append(info); eta = (len(todo) - ok - len(errors)) * (time.time() - t0) / max(ok + len(errors), 1)
                print(f"[{ok + len(errors)}/{len(todo)}] {name} {info:.0f}s on {dev} | elapsed {time.time() - t0:.0f}s, ETA {eta:.0f}s", flush=True)
            else:
                errors.append((name, info)); print(f"[ERROR] {name} on {dev}\\n{info}", flush=True)
    summary = {"ok": ok, "error": len(errors), "skipped": len(jobs) - len(todo), "wall_seconds": time.time() - t0,
               "mean_run_seconds": sum(times) / len(times) if times else None}
    print("grid finished:", summary, flush=True)
    if errors: raise RuntimeError(f"{len(errors)} run(s) failed; first: {errors[0][0]}\\n{errors[0][1]}")
    return summary
"""
open(f"{CODE}/p3lib.py", "w").write(LIB_SRC); open(f"{CODE}/p3parallel.py", "w").write(PAR_SRC)
import sys; sys.path.insert(0, CODE)
import importlib, p3lib as L, p3parallel as PP; importlib.reload(L); importlib.reload(PP)
import torch, numpy as np
log("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(), "| CPU cores:", os.cpu_count())

07:09:47 torch 2.11.0+cu130 | CUDA available: True | GPUs: 2 | CPU cores: 4


In [12]:
# B-2. Integrity checks, then speed test to pick the device (do not mix devices inside one comparison)
assert done("part1_complete"), "Run Part A first."
TAX = L.load_taxonomy(TAX_PATH); DEV = L.load_dev(DEV_DIR)
for seed in SEEDS_ALL:
    st = L.load_stages(PKG[seed]); ok, msgs = L.check_data(st, DEV, TAX)
    log(f"seed {seed}: stage rows {[len(s['y']) for s in st]} | integrity {'OK' if ok else 'FAILED'}")
    assert ok, msgs
    assert all(len(np.unique(s['owner'])) == 5 for s in st), "every stage must have all five clients"
import time
smoke = L.Cfg(learner="DR", seed=101, dose=0.5, rounds=1); st101 = L.load_stages(PKG[101]); timing = {}
for dev_name in ["cpu"] + (["cuda"] if torch.cuda.is_available() else []):
    L.run(L.Cfg(learner="FT", seed=101, rounds=1), st101[:1] + st101[1:], DEV, TAX, threads=1, device=dev_name)   # warm-up
    t0 = time.time(); L.run(smoke, st101, DEV, TAX, threads=1, device=dev_name); timing[dev_name] = time.time() - t0
log("one DR round (all four stages), single process:", {k: round(v, 1) for k, v in timing.items()}, "seconds")
DEVICE = "auto"                       # "cpu", "cuda" or "auto" (cuda only if it measured faster per run)
if DEVICE == "auto": DEVICE = "cuda" if ("cuda" in timing and timing["cuda"] < timing["cpu"]) else "cpu"
WORKERS = 4 if DEVICE == "cuda" else min(os.cpu_count() or 4, 4)
THREADS = 1
QUICK = False                         # True: 2 partitions x 2 replicates, three attack arms (about 15 minutes) for a first test
SEEDS = [101, 102] if QUICK else SEEDS_ALL
REPS = [0, 1] if QUICK else [0, 1, 2, 3]          # training replicates per ownership partition: 3 partitions x 4 = 12 paired units
UNITS = [(s, r) for s in SEEDS for r in REPS]
log(f"DEVICE={DEVICE} WORKERS={WORKERS} QUICK={QUICK} SEEDS={SEEDS}")
CTX = dict(device=DEVICE, threads=THREADS, tax_path=TAX_PATH, dev_dir=DEV_DIR, pkg_dirs={str(s): PKG[s] for s in SEEDS_ALL})
RUNS = f"{OUT2}/runs_{DEVICE}"; os.makedirs(RUNS, exist_ok=True)
json.dump({"device": DEVICE, "workers": WORKERS, "timing_seconds": timing, "torch": torch.__version__, "numpy": np.__version__, "python": sys.version.split()[0],
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "quick": QUICK, "seeds": SEEDS, "reps": REPS}, open(f"{OUT2}/run-environment.json", "w"), indent=1)

07:09:47 seed 101: stage rows [7658, 106239, 7178, 61970] | integrity OK
07:09:48 seed 102: stage rows [7658, 106239, 7178, 61970] | integrity OK
07:09:48 seed 103: stage rows [7658, 106239, 7178, 61970] | integrity OK
07:10:07 one DR round (all four stages), single process: {'cpu': 5.0, 'cuda': 3.6} seconds
07:10:07 DEVICE=cuda WORKERS=4 QUICK=False SEEDS=[101, 102, 103]


In [13]:
# B-3. The experiment grid. Attack arms: k attacking clients, each relabelling `dose` of its Hulk rows as benign in E2 (nested).
Cfg = L.Cfg
CLEAN = (1, 0.0)
ARMS = [CLEAN, (1, 1.0), (3, 1.0), (5, 0.25), (5, 0.5), (5, 0.9)]              # clean, one/three attackers at full dose, all five at partial dose (0.9 keeps Hulk learnable: at 1.0 no Hulk label would survive)
ARMS_Q = [CLEAN, (1, 1.0), (5, 0.9)]
arms = ARMS_Q if QUICK else ARMS
ABL_ARMS = [CLEAN, (5, 0.5)]                                                    # every ablation has a clean twin so damage is measured against its own clean run
G_BASE = [Cfg(learner=l, seed=s, rep=rp, n_att=k, dose=d) for l in ("FT", "EI", "DR") for (s, rp) in UNITS for (k, d) in arms]                        # RQ1
G_DEF = [Cfg(learner="DR", admission=a, seed=s, rep=rp, n_att=k, dose=d) for a in ("DC", "DM", "DQ") for (s, rp) in UNITS for (k, d) in arms]        # RQ2
G_ABL = []                                                                                                                              # RQ3
for (s, rp) in UNITS:
    for (k, d) in ABL_ARMS:
        for kw in (dict(lam_z=0.0), dict(lam_y=0.0), dict(buffer="fifo"), dict(mem=100), dict(mem=1000)): G_ABL.append(Cfg(learner="DR", seed=s, rep=rp, n_att=k, dose=d, **kw))
        for kw in (dict(q=0.5), dict(q=0.9), dict(nmin=5), dict(nmin=100)): G_ABL.append(Cfg(learner="DR", admission="DC", seed=s, rep=rp, n_att=k, dose=d, **kw))
        for v in (10.0, 1000.0): G_ABL.append(Cfg(learner="EI", seed=s, rep=rp, n_att=k, dose=d, lam_E=v))
if QUICK: G_ABL = [c for c in G_ABL if (c.seed, c.rep) == UNITS[0]][:6]
ALL = G_BASE + G_DEF + G_ABL
log("ownership partitions", len(SEEDS), "x replicates", len(REPS), "=", len(UNITS), "units |" "| arms", arms, "| runs: learners", len(G_BASE), "| defences", len(G_DEF), "| ablations", len(G_ABL), "| total", len(ALL))

07:10:07 ownership partitions 3 x replicates 4 = 12 units || arms [(1, 0.0), (1, 1.0), (3, 1.0), (5, 0.25), (5, 0.5), (5, 0.9)] | runs: learners 216 | defences 216 | ablations 264 | total 696


In [14]:
# B-4. RQ1: FT / EI / DR under clean and poisoned conditions
s1 = PP.run_grid(G_BASE, RUNS, CTX, workers=WORKERS)

216 runs in the grid, 0 already finished, 216 to run on 4 worker(s) [cuda]
[1/216] FT-NONE-d100-k1-r0-s101.json 23s on cuda:0 | elapsed 26s, ETA 5566s
[2/216] FT-NONE-d000-k1-r0-s101.json 23s on cuda:0 | elapsed 26s, ETA 2785s
[3/216] FT-NONE-d025-k5-r0-s101.json 23s on cuda:1 | elapsed 26s, ETA 1853s
[4/216] FT-NONE-d100-k3-r0-s101.json 23s on cuda:1 | elapsed 26s, ETA 1384s
[5/216] FT-NONE-d050-k5-r0-s101.json 20s on cuda:0 | elapsed 46s, ETA 1948s
[6/216] FT-NONE-d000-k1-r1-s101.json 20s on cuda:1 | elapsed 46s, ETA 1624s
[7/216] FT-NONE-d090-k5-r0-s101.json 20s on cuda:0 | elapsed 46s, ETA 1388s
[8/216] FT-NONE-d100-k1-r1-s101.json 21s on cuda:1 | elapsed 47s, ETA 1212s
[9/216] FT-NONE-d100-k3-r1-s101.json 20s on cuda:0 | elapsed 66s, ETA 1523s
[10/216] FT-NONE-d025-k5-r1-s101.json 20s on cuda:1 | elapsed 67s, ETA 1370s
[11/216] FT-NONE-d090-k5-r1-s101.json 20s on cuda:1 | elapsed 67s, ETA 1244s
[12/216] FT-NONE-d050-k5-r1-s101.json 20s on cuda:0 | elapsed 67s, ETA 1136s
[13/216] F

In [15]:
# B-5. RQ2: admission rules DC / DM / DQ on top of DR
s2 = PP.run_grid(G_DEF, RUNS, CTX, workers=WORKERS)

216 runs in the grid, 0 already finished, 216 to run on 4 worker(s) [cuda]
[1/216] DR-DC-d100-k3-r0-s101.json 50s on cuda:1 | elapsed 53s, ETA 11389s
[2/216] DR-DC-d025-k5-r0-s101.json 53s on cuda:1 | elapsed 56s, ETA 6001s
[3/216] DR-DC-d100-k1-r0-s101.json 54s on cuda:0 | elapsed 57s, ETA 4024s
[4/216] DR-DC-d000-k1-r0-s101.json 55s on cuda:0 | elapsed 58s, ETA 3071s
[5/216] DR-DC-d050-k5-r0-s101.json 49s on cuda:1 | elapsed 102s, ETA 4308s
[6/216] DR-DC-d090-k5-r0-s101.json 46s on cuda:1 | elapsed 102s, ETA 3576s
[7/216] DR-DC-d000-k1-r1-s101.json 52s on cuda:0 | elapsed 108s, ETA 3237s
[8/216] DR-DC-d100-k1-r1-s101.json 51s on cuda:0 | elapsed 109s, ETA 2829s
[9/216] DR-DC-d100-k3-r1-s101.json 50s on cuda:1 | elapsed 152s, ETA 3495s
[10/216] DR-DC-d025-k5-r1-s101.json 51s on cuda:1 | elapsed 153s, ETA 3146s
[11/216] DR-DC-d090-k5-r1-s101.json 47s on cuda:0 | elapsed 156s, ETA 2902s
[12/216] DR-DC-d050-k5-r1-s101.json 49s on cuda:0 | elapsed 157s, ETA 2671s
[13/216] DR-DC-d100-k1-r2

In [16]:
# B-6. RQ3: ablations (replay terms, memory eviction and size, admission quota and small-class threshold, EWC strength), each with its clean twin
s3 = PP.run_grid(G_ABL, RUNS, CTX, workers=WORKERS)

264 runs in the grid, 0 already finished, 264 to run on 4 worker(s) [cuda]
[1/264] DR-NONE-d000-k1-r0-s101-z0.json 42s on cuda:1 | elapsed 45s, ETA 11871s
[2/264] DR-NONE-d000-k1-r0-s101-y0.json 47s on cuda:0 | elapsed 50s, ETA 6524s
[3/264] DR-NONE-d000-k1-r0-s101-fifo.json 59s on cuda:1 | elapsed 63s, ETA 5446s
[4/264] DR-NONE-d000-k1-r0-s101-mem100.json 60s on cuda:0 | elapsed 63s, ETA 4100s
[5/264] DR-NONE-d000-k1-r0-s101-mem1000.json 57s on cuda:1 | elapsed 102s, ETA 5272s
[6/264] DR-DC-d000-k1-r0-s101-q0.5.json 56s on cuda:0 | elapsed 105s, ETA 4533s
[7/264] DR-DC-d000-k1-r0-s101-nmin5.json 57s on cuda:0 | elapsed 120s, ETA 4401s
[8/264] DR-DC-d000-k1-r0-s101-q0.9.json 57s on cuda:1 | elapsed 120s, ETA 3840s
[9/264] EI-NONE-d000-k1-r0-s101-lam_E10.json 41s on cuda:0 | elapsed 146s, ETA 4145s
[10/264] DR-DC-d000-k1-r0-s101-nmin100.json 57s on cuda:1 | elapsed 159s, ETA 4029s
[11/264] DR-NONE-d050-k5-r0-s101-z0.json 41s on cuda:1 | elapsed 161s, ETA 3701s
[12/264] EI-NONE-d000-k1-r

In [17]:
# B-7. Check that every run finished with sane numbers, then PART B package
import glob, zipfile
files = sorted(glob.glob(f"{RUNS}/*.json")); assert len(files) >= len(ALL), (len(files), len(ALL))
bad = []
for p in files:
    r = json.load(open(p)); f1 = [s["macro_f1"] for s in r["stages"]]
    if not all(np.isfinite(f1)) or [len(s["slots"]) for s in r["stages"]] != [3, 8, 13, 16] or r["device"].split(":")[0] != DEVICE: bad.append(os.path.basename(p))
log("run files:", len(files), "| failing sanity checks:", bad); assert not bad
with zipfile.ZipFile(f"{W}/phase3-part2-outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT2):
        for fn in fs: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), W))
mark("part2_complete"); log("PART B COMPLETE -> /kaggle/working/phase3-part2-outputs.zip", round(os.path.getsize(f"{W}/phase3-part2-outputs.zip") / 1e6, 1), "MB")

09:25:17 run files: 696 | failing sanity checks: []
09:25:17 PART B COMPLETE -> /kaggle/working/phase3-part2-outputs.zip 1.9 MB


# PART C: results, statistics, figures, comparison and discussion (reads the saved runs only; no training)

In [18]:
# C-1. Load every run into one table; damage is always measured against the same learner/variant/seed without attack
import glob, json, os, csv, math, hashlib, numpy as np, pandas as pd
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "matplotlib"]); import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
dirs = sorted(glob.glob(f"{OUT2}/runs_*")); assert dirs, "No runs found: run Part B (or unzip phase3-part2-outputs.zip into /kaggle/working)."
RUNS_DIR = dirs[0] if len(dirs) == 1 else max(dirs, key=lambda d: len(os.listdir(d))); log("using runs from", RUNS_DIR)
FIG = f"{OUT3}/figures"; os.makedirs(FIG, exist_ok=True)
rows = []; used = []
for p in sorted(glob.glob(f"{RUNS_DIR}/*.json")):
    r = json.load(open(p)); c = r["cfg"]; st = r["stages"]; used.append(p); po = r["poison"]
    hk = lambda i: st[i].get("hulk_recall", np.nan); asr = lambda i: st[i].get("hulk_asr", np.nan)
    var = r["tag"].split(f"-s{c['seed']}")[1].lstrip("-") or "default"
    a2 = r["admission"][1]
    share = (po["flipped"] / po["hulk_rows_stage"]) if c["dose"] > 0 and po.get("hulk_rows_stage") else 0.0
    rows.append(dict(tag=r["tag"], learner=c["learner"], adm=c["admission"], dose=c["dose"], k=c["n_att"] if c["dose"] > 0 else 0, seed=c["seed"], rep=c["rep"], unit=f"{c['seed']}-{c['rep']}", variant=var,
        arm="clean" if c["dose"] == 0 else f"k{c['n_att']}-d{c['dose']:g}", share=share,
        f1_E1=st[0]["macro_f1"], f1_E2=st[1]["macro_f1"], f1_E3=st[2]["macro_f1"], f1_E4=st[3]["macro_f1"],
        far_E1=st[0]["benign_far"], far_E2=st[1]["benign_far"], far_E3=st[2]["benign_far"], far_E4=st[3]["benign_far"],
        hulk_E2=hk(1), hulk_E3=hk(2), hulk_E4=hk(3), asr_E2=asr(1), asr_E4=asr(3),
        A=r["summary"]["A"], BWT=r["summary"]["BWT"], forgetting=r["summary"]["forgetting"],
        seconds=r["cost"]["seconds"], model_bytes=r["cost"]["model_bytes"], comm_bytes=r["cost"]["comm_bytes_total"], buffer_bytes=r["cost"]["buffer_bytes_total"],
        buf_poison_end=r["cost"]["buffer_poisoned_records_end"], poisoned_rows=a2["poisoned_rows"], adm_poison=a2["admitted_poisoned"],
        adm_frac=(a2["admitted"] / a2["rows"]) if a2["rows"] else np.nan, newcls_kept=(a2["new_class_rows_admitted"] / a2["new_class_rows"]) if a2["new_class_rows"] else np.nan))
df = pd.DataFrame(rows)
key = ["learner", "adm", "variant", "unit"]
clean = df[df.dose == 0].set_index(key)
def paired(r):
    k = tuple(r[x] for x in key)
    if k not in clean.index: return pd.Series({"H": np.nan, "dRec": np.nan, "dFAR": np.nan, "dF1": np.nan, "dASR": np.nan})
    c = clean.loc[k]
    return pd.Series({"H": c.hulk_E4 - r.hulk_E4, "dRec": c.hulk_E2 - r.hulk_E2, "dFAR": r.far_E4 - c.far_E4, "dF1": r.f1_E4 - c.f1_E4, "dASR": r.asr_E4 - c.asr_E4})
df = pd.concat([df, df.apply(paired, axis=1)], axis=1)
ARM_ORDER = sorted(df.arm.unique(), key=lambda a: (df[df.arm == a].share.mean(), a))
df.to_csv(f"{OUT3}/all-runs.csv", index=False)
log("runs:", len(df), "| units (ownership partition x replicate)", df.unit.nunique(), "| arms (by share of Hulk rows poisoned):", [(a, round(df[df.arm == a].share.mean(), 3)) for a in ARM_ORDER])
def ms(x): x = np.asarray(x, float); x = x[~np.isnan(x)]; return (float(x.mean()), float(x.std(ddof=1)) if len(x) > 1 else 0.0, len(x))
def savefig(fig, name, header, data):
    fig.tight_layout(); fig.savefig(f"{FIG}/{name}.png", dpi=160); fig.savefig(f"{FIG}/{name}.pdf"); plt.close(fig)
    with open(f"{OUT3}/{name}.csv", "w", newline="") as f: w = csv.writer(f); w.writerow(header); w.writerows(data)
COL = {"FT": "tab:blue", "EI": "tab:orange", "DR": "tab:green", "NONE": "tab:green", "DC": "tab:red", "DM": "tab:purple", "DQ": "tab:gray"}
XS = {a: df[df.arm == a].share.mean() for a in ARM_ORDER}

09:25:19 using runs from /kaggle/working/outputs/part2/runs_cuda
09:25:19 runs: 696 | units (ownership partition x replicate) 12 | arms (by share of Hulk rows poisoned): [('clean', np.float64(0.0)), ('k5-d0.25', np.float64(0.25)), ('k1-d1', np.float64(0.257)), ('k5-d0.5', np.float64(0.5)), ('k3-d1', np.float64(0.703)), ('k5-d0.9', np.float64(0.9))]


In [19]:
# C-2. Baseline evaluation (requirement 3): clean runs of FT / EI / DR, mean and sd over seeds
base = df[(df.dose == 0) & (df.adm == "NONE") & (df.variant == "default")]
tab = []
for L_ in ("FT", "EI", "DR"):
    d = base[base.learner == L_]
    if d.empty: continue
    tab.append([L_, len(d)] + [x for m in ("f1_E4", "far_E4", "hulk_E4", "A", "BWT", "forgetting", "seconds") for x in ms(d[m])[:2]])
hdr = ["learner", "n", "macroF1_E4", "sd", "FAR_E4", "sd", "Hulk_recall_E4", "sd", "A_new_classes", "sd", "BWT", "sd", "forgetting", "sd", "seconds", "sd"]
with open(f"{OUT3}/table-baselines.csv", "w", newline="") as f: w = csv.writer(f); w.writerow(hdr); w.writerows(tab)
print(pd.DataFrame(tab, columns=hdr).round(4).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(10, 4)); data = []
for L_ in ("FT", "EI", "DR"):
    d = base[base.learner == L_]
    if d.empty: continue
    m = [ms(d[f"f1_E{t}"]) for t in range(1, 5)]; mf = [ms(d[f"far_E{t}"]) for t in range(1, 5)]
    ax[0].errorbar(range(1, 5), [a for a, _, _ in m], [b for _, b, _ in m], marker="o", capsize=3, color=COL[L_], label=L_)
    ax[1].errorbar(range(1, 5), [a for a, _, _ in mf], [b for _, b, _ in mf], marker="o", capsize=3, color=COL[L_], label=L_)
    data += [[L_, t, *m[t-1][:2], *mf[t-1][:2]] for t in range(1, 5)]
ax[0].set(xlabel="stage", ylabel="macro-F1 (eligible classes)", title="Clean macro-F1 (mean +/- sd over seeds)", xticks=range(1, 5)); ax[1].set(xlabel="stage", ylabel="benign false-alarm rate", title="Clean benign FAR", xticks=range(1, 5))
for a in ax: a.grid(alpha=.3); a.legend()
savefig(fig, "fig1-clean-by-stage", ["learner", "stage", "macroF1_mean", "macroF1_sd", "FAR_mean", "FAR_sd"], data)
fig, ax = plt.subplots(1, 3, figsize=(11, 3.6)); data = []
for i, (mname, ttl) in enumerate([("A", "Acquisition A (new classes)"), ("BWT", "Backward transfer"), ("forgetting", "Forgetting")]):
    vals = [ms(base[base.learner == L_][mname]) for L_ in ("FT", "EI", "DR")]
    ax[i].bar(["FT", "EI", "DR"], [v[0] for v in vals], yerr=[v[1] for v in vals], color=[COL[x] for x in ("FT", "EI", "DR")], capsize=4); ax[i].set_title(ttl); ax[i].grid(axis="y", alpha=.3)
    data += [[mname, L_, v[0], v[1]] for L_, v in zip(("FT", "EI", "DR"), vals)]
savefig(fig, "fig2-retention-acquisition", ["metric", "learner", "mean", "sd"], data)

learner  n  macroF1_E4     sd  FAR_E4     sd  Hulk_recall_E4     sd  A_new_classes     sd     BWT     sd  forgetting     sd  seconds     sd
     FT 12      0.0308 0.0116  0.9845 0.0292          0.0000 0.0000         0.1844 0.0345 -0.7190 0.0936      0.7190 0.0936  20.3306 0.9039
     EI 12      0.0461 0.0282  0.9315 0.1379          0.0000 0.0000         0.1862 0.0651 -0.6883 0.1104      0.6883 0.1104  37.9720 0.6749
     DR 12      0.4907 0.0695  0.1038 0.0396          0.9934 0.0064         0.2986 0.0668 -0.0474 0.1581      0.1922 0.1196  53.1434 0.6520


In [20]:
# C-3. RQ1: poisoning dose-response. x = share of all E2 Hulk rows that were relabelled benign (one point per attack arm)
nd = df[(df.adm == "NONE") & (df.variant == "default")]
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8)); data = []
PANELS = [("dRec", "Hulk recall lost at the poisoned stage (E2)"), ("H", "Persistent damage H (Hulk recall lost at E4)"), ("asr_E4", "Hulk predicted benign at E4 (ASR)"), ("dFAR", "Benign FAR increase at E4")]
for L_ in ("FT", "EI", "DR"):
    d = nd[nd.learner == L_]
    for i, (m, ttl) in enumerate(PANELS):
        v = [ms(d[d.arm == a][m]) for a in ARM_ORDER if a != "clean"]; x = [XS[a] for a in ARM_ORDER if a != "clean"]
        ax[i].errorbar(x, [a for a, _, _ in v], [b for _, b, _ in v], marker="o", capsize=3, color=COL[L_], label=L_); ax[i].set_title(ttl, fontsize=8)
        data += [[L_, m, a, XS[a], p, q] for a, (p, q, _) in zip([a for a in ARM_ORDER if a != "clean"], v)]
for a in ax: a.set_xlabel("share of Hulk rows poisoned"); a.grid(alpha=.3); a.legend()
savefig(fig, "fig3-dose-response", ["learner", "metric", "arm", "share_poisoned", "mean", "sd"], data)
tt = []
for L_ in ("FT", "EI", "DR"):
    for a in ARM_ORDER:
        d = nd[(nd.learner == L_) & (nd.arm == a)]
        if len(d): tt.append([L_, a, round(XS[a], 3), len(d)] + [y for m in ("hulk_E2", "hulk_E4", "dRec", "H", "asr_E4", "far_E4", "f1_E4", "BWT") for y in ms(d[m])[:2]])
th = ["learner", "arm", "share_poisoned", "n", "Hulk_E2", "sd", "Hulk_E4", "sd", "dRec", "sd", "H", "sd", "ASR_E4", "sd", "FAR_E4", "sd", "macroF1_E4", "sd", "BWT", "sd"]
with open(f"{OUT3}/table-dose-response.csv", "w", newline="") as f: w = csv.writer(f); w.writerow(th); w.writerows(tt)
print(pd.DataFrame(tt, columns=th).round(3).to_string(index=False))

learner      arm  share_poisoned  n  Hulk_E2    sd  Hulk_E4    sd  dRec    sd     H    sd  ASR_E4    sd  FAR_E4    sd  macroF1_E4    sd    BWT    sd
     FT    clean           0.000 12    1.000 0.000    0.000 0.000 0.000 0.000 0.000 0.000   0.000 0.000   0.984 0.029       0.031 0.012 -0.719 0.094
     FT k5-d0.25           0.250 12    1.000 0.000    0.000 0.000 0.000 0.000 0.000 0.000   0.000 0.000   1.000 0.000       0.028 0.014 -0.724 0.093
     FT    k1-d1           0.257 12    0.968 0.069    0.000 0.000 0.031 0.069 0.000 0.000   0.000 0.000   0.999 0.004       0.031 0.013 -0.711 0.089
     FT  k5-d0.5           0.500 12    0.506 0.458    0.000 0.000 0.494 0.458 0.000 0.000   0.000 0.000   1.000 0.000       0.028 0.012 -0.684 0.115
     FT    k3-d1           0.703 12    0.165 0.364    0.000 0.000 0.835 0.364 0.000 0.000   0.000 0.000   1.000 0.000       0.030 0.013 -0.630 0.104
     FT  k5-d0.9           0.900 12    0.000 0.000    0.000 0.000 1.000 0.000 0.000 0.000   0.000 0.000   

In [21]:
# C-4. RQ2: admission rules versus no admission (DR learner)
dr = df[(df.learner == "DR") & (df.variant == "default")]
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8)); data = []
for a_ in ("NONE", "DC", "DM", "DQ"):
    d = dr[dr.adm == a_]
    for i, (m, ttl) in enumerate([("dRec", "Hulk recall lost at E2"), ("H", "Persistent damage H"), ("A", "Acquisition A (new classes)"), ("f1_E4", "Macro-F1 at E4")]):
        arms_ = ARM_ORDER if m in ("A", "f1_E4") else [a for a in ARM_ORDER if a != "clean"]
        v = [ms(d[d.arm == a][m]) for a in arms_]
        ax[i].errorbar([XS[a] for a in arms_], [p for p, _, _ in v], [q for _, q, _ in v], marker="o", capsize=3, color=COL[a_], label=a_); ax[i].set_title(ttl, fontsize=9)
        data += [[a_, m, a, XS[a], p, q] for a, (p, q, _) in zip(arms_, v)]
for a in ax: a.set_xlabel("share of Hulk rows poisoned"); a.grid(alpha=.3); a.legend()
savefig(fig, "fig4-defences", ["admission", "metric", "arm", "share_poisoned", "mean", "sd"], data)
at = []; strongest = max((a for a in ARM_ORDER), key=lambda a: XS[a])
for a_ in ("NONE", "DC", "DM", "DQ"):
    d = dr[(dr.adm == a_) & (dr.arm == strongest)]
    if d.empty: continue
    at.append([a_, len(d)] + [y for m in ("adm_frac", "newcls_kept", "buf_poison_end", "dRec", "H", "A", "f1_E4") for y in ms(d[m])[:2]] + [float(np.mean(d.adm_poison / d.poisoned_rows))])
ah = ["rule", "n", "admitted_fraction", "sd", "new_class_rows_kept", "sd", "poisoned_records_in_buffers_end", "sd", "dRec", "sd", "H", "sd", "A", "sd", "macroF1_E4", "sd", "poisoned_rows_admitted_fraction"]
with open(f"{OUT3}/table-admission.csv", "w", newline="") as f: w = csv.writer(f); w.writerow(ah); w.writerows(at)
print("strongest arm:", strongest); print(pd.DataFrame(at, columns=ah).round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3.6)); ax.bar([r[0] for r in at], [r[-1] for r in at], color=[COL[r[0]] for r in at]); ax.set_ylabel("share of poisoned rows admitted (E2)"); ax.set_title(f"Poison admitted by each rule ({strongest})"); ax.grid(axis="y", alpha=.3)
savefig(fig, "fig5-poison-admitted", ["rule", "share_poisoned_rows_admitted"], [[r[0], r[-1]] for r in at])

strongest arm: k5-d0.9
rule  n  admitted_fraction  sd  new_class_rows_kept   sd  poisoned_records_in_buffers_end     sd  dRec    sd     H    sd     A    sd  macroF1_E4    sd  poisoned_rows_admitted_fraction
NONE 12               1.00 0.0                1.000 0.00                         1114.667 56.378 1.000 0.000 0.993 0.006 0.299 0.067       0.393 0.063                            1.000
  DC 12               0.79 0.0                1.000 0.00                         1036.750 71.284 0.974 0.041 0.988 0.024 0.175 0.071       0.275 0.102                            0.775
  DM 12               0.79 0.0                0.095 0.04                         1265.000 93.275 1.000 0.000 0.995 0.006 0.140 0.050       0.257 0.097                            0.940
  DQ 12               0.79 0.0                1.000 0.00                          992.250 85.328 1.000 0.000 0.994 0.008 0.269 0.068       0.401 0.100                            0.750


In [22]:
# C-5. RQ3 / ablation studies (requirement 6): every variant is compared with the same arm of its own reference, damage against its own clean twin
def arm_row(l, adm, var, arm_):
    d = df[(df.learner == l) & (df.adm == adm) & (df.variant == var) & (df.arm == arm_)]
    return d
AB = [("DR reference", "DR", "NONE", "default"), ("DR: no logit-matching (lam_z=0)", "DR", "NONE", "z0"), ("DR: no replay-label term (lam_y=0)", "DR", "NONE", "y0"), ("DR: FIFO memory", "DR", "NONE", "fifo"),
      ("DR: memory 100/client", "DR", "NONE", "mem100"), ("DR: memory 1000/client", "DR", "NONE", "mem1000"),
      ("DR+DC reference", "DR", "DC", "default"), ("DR+DC: quota q=0.5", "DR", "DC", "q0.5"), ("DR+DC: quota q=0.9", "DR", "DC", "q0.9"), ("DR+DC: small-class threshold 5", "DR", "DC", "nmin5"), ("DR+DC: small-class threshold 100", "DR", "DC", "nmin100"),
      ("EI reference (lam_E=100)", "EI", "NONE", "default"), ("EI: lam_E=10", "EI", "NONE", "lam_E10"), ("EI: lam_E=1000", "EI", "NONE", "lam_E1000")]
AB_ARM = "k5-d0.5"; ab = []
for name, l, adm, var in AB:
    p = arm_row(l, adm, var, AB_ARM); c0 = arm_row(l, adm, var, "clean")
    if p.empty: continue
    ab.append([name, len(p)] + [y for m in ("dRec", "H", "hulk_E4", "f1_E4", "far_E4", "A", "BWT", "buf_poison_end") for y in ms(p[m])[:2]] + [ms(c0["hulk_E4"])[0], ms(c0["f1_E4"])[0]])
ah = ["arm", "n", "dRec", "sd", "H", "sd", "Hulk_E4", "sd", "macroF1_E4", "sd", "FAR_E4", "sd", "A", "sd", "BWT", "sd", "poisoned_buffer_records", "sd", "clean_Hulk_E4", "clean_macroF1_E4"]
with open(f"{OUT3}/table-ablations.csv", "w", newline="") as f: w = csv.writer(f); w.writerow(ah); w.writerows(ab)
print(f"ablations at {AB_ARM}"); print(pd.DataFrame(ab, columns=ah).round(3).to_string(index=False))
fig, ax = plt.subplots(1, 3, figsize=(15, 4.8)); y = np.arange(len(ab))
for i, (idx, ttl, colr) in enumerate([(2, "Hulk recall lost at E2 (lower is better)", "tab:red"), (4, "Persistent damage H (lower is better)", "tab:orange"), (8, "Macro-F1 at E4 (higher is better)", "tab:green")]):
    ax[i].barh(y, [r[idx] for r in ab], xerr=[r[idx + 1] for r in ab], capsize=2, color=colr); ax[i].set_title(ttl, fontsize=9); ax[i].invert_yaxis(); ax[i].grid(axis="x", alpha=.3)
    ax[i].set_yticks(y); ax[i].set_yticklabels([r[0] for r in ab] if i == 0 else [], fontsize=7)
savefig(fig, "fig6-ablations", ah, ab)

ablations at k5-d0.5
                               arm  n  dRec    sd     H    sd  Hulk_E4    sd  macroF1_E4    sd  FAR_E4    sd     A    sd    BWT    sd  poisoned_buffer_records     sd  clean_Hulk_E4  clean_macroF1_E4
                      DR reference 12 0.612 0.313 0.785 0.209    0.209 0.212       0.427 0.052   0.087 0.038 0.298 0.062 -0.084 0.183                  630.000 32.694          0.993             0.491
   DR: no logit-matching (lam_z=0) 12 0.430 0.286 0.637 0.249    0.353 0.254       0.401 0.063   0.156 0.063 0.270 0.059 -0.125 0.172                  630.000 32.694          0.990             0.453
DR: no replay-label term (lam_y=0) 12 0.883 0.288 0.000 0.000    0.000 0.000       0.055 0.035   0.789 0.237 0.189 0.058 -0.603 0.150                  630.000 32.694          0.000             0.083
                   DR: FIFO memory 12 0.561 0.420 0.000 0.000    0.000 0.000       0.056 0.041   0.791 0.284 0.204 0.051 -0.601 0.223                    0.000  0.000          0.000   

In [23]:
# C-6. Cost (time, communication, memory)
c = df[(df.dose == 0) & (df.adm == "NONE") & (df.variant == "default")].groupby("learner")[["seconds", "model_bytes", "comm_bytes", "buffer_bytes"]].mean()
c["comm_MB"] = c.comm_bytes / 1e6; c["buffer_KB"] = c.buffer_bytes / 1e3; c["model_KB"] = c.model_bytes / 1e3
cd = df[(df.dose == 0) & (df.learner == "DR") & (df.variant == "default")].groupby("adm")["seconds"].mean()
c.to_csv(f"{OUT3}/table-cost.csv"); print(c[["seconds", "model_KB", "comm_MB", "buffer_KB"]].round(2).to_string()); print("DR seconds by admission rule:", cd.round(1).to_dict())
fig, ax = plt.subplots(1, 3, figsize=(11, 3.4))
for i, (m, ttl) in enumerate([("seconds", "Seconds per run"), ("comm_MB", "Communication per run (MB)"), ("buffer_KB", "Replay memory (KB, all clients)")]):
    ax[i].bar(c.index, c[m], color=[COL[x] for x in c.index]); ax[i].set_title(ttl); ax[i].grid(axis="y", alpha=.3)
savefig(fig, "fig7-cost", ["learner", "seconds", "comm_MB", "buffer_KB"], c.reset_index()[["learner", "seconds", "comm_MB", "buffer_KB"]].values.tolist())

         seconds  model_KB  comm_MB  buffer_KB
learner                                       
DR         53.14     75.58    35.53      940.0
EI         37.97     75.58    35.53        0.0
FT         20.33     75.58    35.53        0.0
DR seconds by admission rule: {'DC': 52.8, 'DM': 52.5, 'DQ': 53.1, 'NONE': 53.1}


In [24]:
# C-7. Confirmatory statistics exactly as planned in Phase 2: paired by seed, two-sided Wilcoxon signed-rank, Holm correction inside each family,
#      mean/median difference, rank-biserial effect size, 95% bootstrap CI over whole seeds (10,000 resamples)
from scipy import stats
rng_b = np.random.default_rng(20260101)
def seedvec(mask, m): return df[mask].groupby("unit")[m].mean()
def test(a, b):
    d = (a - b).dropna(); n = len(d)
    if n < 3: return None
    nz = d[d != 0]
    p = 1.0 if len(nz) == 0 else float(stats.wilcoxon(d, zero_method="wilcox", alternative="two-sided", method="auto").pvalue)
    if len(nz):
        rk = stats.rankdata(nz.abs()); wp = rk[(nz > 0).values].sum(); wm = rk[(nz < 0).values].sum(); rbc = float((wp - wm) / (wp + wm))
    else: rbc = 0.0
    bs = [d.values[rng_b.integers(0, n, n)].mean() for _ in range(10000)]
    return dict(n=n, mean_diff=float(d.mean()), median_diff=float(d.median()), ci_lo=float(np.percentile(bs, 2.5)), ci_hi=float(np.percentile(bs, 97.5)), rank_biserial=rbc, p=p)
def holm(ps):
    order = np.argsort(ps); m = len(ps); adj = np.empty(m); run_max = 0.0
    for i, j in enumerate(order): run_max = max(run_max, (m - i) * ps[j]); adj[j] = min(1.0, run_max)
    return adj
M = lambda l, adm, arm_: (df.learner == l) & (df.adm == adm) & (df.variant == "default") & (df.arm == arm_)
nonclean = [a for a in ARM_ORDER if a != "clean"]; fam = []
for e in ("H", "dRec"):
    for l1, l2 in (("DR", "EI"), ("DR", "FT"), ("EI", "FT")):
        for a in nonclean:
            t = test(seedvec(M(l1, "NONE", a), e), seedvec(M(l2, "NONE", a), e))
            if t: fam.append(dict(family="C1", test=f"{l1} minus {l2}", endpoint=e, arm=a, **t))
for a in ARM_ORDER:
    t = test(seedvec(M("DR", "DC", a), "A"), seedvec(M("DR", "DM", a), "A"))
    if t: fam.append(dict(family="C2", test="DC minus DM", endpoint="A (acquisition)", arm=a, **t))
for e in ("H", "dRec"):
    for a in nonclean:
        t = test(seedvec(M("DR", "DC", a), e), seedvec(M("DR", "DM", a), e))
        if t: fam.append(dict(family="C2", test="DC minus DM", endpoint=e, arm=a, **t))
t = test(seedvec(M("DR", "DC", "clean"), "far_E4"), seedvec(M("DR", "DM", "clean"), "far_E4"))
if t: fam.append(dict(family="C2", test="DC minus DM", endpoint="clean FAR (E4)", arm="clean", **t))
conf = pd.DataFrame(fam)
for f_ in conf.family.unique():
    ix = conf.family == f_; conf.loc[ix, "p_holm"] = holm(conf.loc[ix, "p"].values)
conf["significant_holm_0.05"] = conf.p_holm < 0.05
conf.to_csv(f"{OUT3}/table-confirmatory-tests.csv", index=False)
print(conf.groupby("family").agg(tests=("p", "size"), significant=("significant_holm_0.05", "sum")).to_string())
print(conf[conf["significant_holm_0.05"]].round(4).to_string(index=False)[:6000])
# exploratory: every admission rule and learner against its reference, same arms (descriptive paired differences with min/max over seeds)
exp = []
for a in ARM_ORDER:
    for e in ("H", "dRec", "f1_E4", "far_E4", "A"):
        for adm in ("DC", "DM", "DQ"):
            if a == "clean" and e in ("H", "dRec"): continue
            d = (seedvec(M("DR", adm, a), e) - seedvec(M("DR", "NONE", a), e)).dropna()
            if len(d): exp.append([f"{adm} minus NONE", a, e, len(d), float(d.min()), float(d.mean()), float(d.max()), int((d > 0).sum())])
pd.DataFrame(exp, columns=["comparison", "arm", "metric", "n_seeds", "min", "mean", "max", "seeds_positive"]).to_csv(f"{OUT3}/table-exploratory-differences.csv", index=False)

        tests  significant
family                    
C1         30            6
C2         17            6
family        test        endpoint      arm  n  mean_diff  median_diff   ci_lo   ci_hi  rank_biserial      p  p_holm  significant_holm_0.05
    C1 DR minus EI               H k5-d0.25 12     0.0317       0.0129  0.0120  0.0564         0.9744 0.0010  0.0254                   True
    C1 DR minus EI               H  k5-d0.5 12     0.7846       0.8577  0.6681  0.8906         1.0000 0.0005  0.0146                   True
    C1 DR minus EI               H  k5-d0.9 12     0.9933       0.9959  0.9898  0.9966         1.0000 0.0005  0.0146                   True
    C1 DR minus FT               H k5-d0.25 12     0.0317       0.0129  0.0122  0.0569         0.9744 0.0010  0.0254                   True
    C1 DR minus FT               H  k5-d0.5 12     0.7846       0.8577  0.6656  0.8929         1.0000 0.0005  0.0146                   True
    C1 DR minus FT               H  k5-d0.9 12     0

In [25]:
# C-8. Comparison with existing research (requirement 5)
refs = [("FT (sequential fine-tuning)", "plain continual-learning lower bound", "forgetting is expected; baseline everywhere"),
        ("EI = EWC-style regularisation", "Kirkpatrick et al., 2017 (EWC); per-example diagonal Fisher here", "regularisation-based retention without stored data"),
        ("DR = DER++ replay", "Buzzega et al., 2020 (DER++)", "memory of raw rows plus logits, per client (500 records)"),
        ("FedAvg aggregation", "McMahan et al., 2017", "weighted by admitted rows, 12 rounds per stage"),
        ("Label-flipping poisoning", "standard data-poisoning threat model in federated learning", "1, 3 or 5 clients relabel Hulk as benign in E2, then withdraw"),
        ("Admission rules DC / DM / DQ", "this project (class-aware quota, matched-volume ranking, random control)", "tests whether protection can avoid blocking legitimate new classes")]
with open(f"{OUT3}/table-literature-mapping.csv", "w", newline="") as f: w = csv.writer(f); w.writerow(["component", "literature", "role here"]); w.writerows(refs)
papers = sorted(os.listdir(f"{R}/docs/papers")) if os.path.isdir(f"{R}/docs/papers") else []
open(f"{OUT3}/phase2-paper-files.txt", "w").write("\n".join(papers)); print(len(papers), "Phase 2 paper files listed in the repository (docs/papers)")
cmp_rows = []
try:
    sel = json.load(open(f"{R}/results/phase3-cpu-v3/development/tuning-selection.json"))
    for L_ in ("FT", "EI", "DR"):
        vals = []
        for i, seed in enumerate([101, 102, 103]):
            row = sel["learners"][L_]["selected"]["rows"][i]
            phys = json.load(open(f"{R}/results/phase3-cpu-v3/development/receipts/{row}.json"))["row"].get("alias_of") or row
            h = sorted(glob.glob(f"{R}/results/phase3-cpu-v3/development/executions/{phys}/attempt-*/evaluation-*/metrics.json"))
            if h: vals.append(json.load(open(h[-1]))["stages"][-1]["eligible_macro_f1"])
        ours = base[(base.learner == L_) & (base.rep == 0)]["f1_E4"]
        cmp_rows.append([L_, len(vals), float(np.mean(vals)) if vals else np.nan, float(ours.mean()) if len(ours) else np.nan])
except Exception as e:
    log("committed-results cross-check skipped:", repr(e)[:200])
with open(f"{OUT3}/table-crosscheck-repository.csv", "w", newline="") as f: w = csv.writer(f); w.writerow(["learner", "n_committed", "macroF1_E4_committed", "macroF1_E4_this_implementation"]); w.writerows(cmp_rows)
print(pd.DataFrame(cmp_rows, columns=["learner", "n_committed", "committed", "this implementation"]).round(3).to_string(index=False))

19 Phase 2 paper files listed in the repository (docs/papers)
learner  n_committed  committed  this implementation
     FT            3      0.018                0.033
     EI            3      0.037                0.053
     DR            3      0.655                0.434


In [26]:
# C-9. Analysis and discussion (generated from the numbers; limits stated), requirement matrix, provenance, PART C package
V = lambda l, adm, arm_, m, var="default": ms(df[(df.learner == l) & (df.adm == adm) & (df.variant == var) & (df.arm == arm_)][m])
out = ["# Results analysis and discussion", "", f"{df.unit.nunique()} paired units (3 ownership partitions x training replicates), paired by unit; confirmatory tests follow the Phase 2 plan (Wilcoxon signed-rank, Holm inside each family). Development bank only. Device: {os.path.basename(RUNS_DIR)}.", ""]
out.append("## 1. Clean baselines")
for l in ("FT", "EI", "DR"):
    a, s, n = V(l, "NONE", "clean", "f1_E4"); out.append(f"- {l}: final macro-F1 {a:.3f} (sd {s:.3f}), backward transfer {V(l, 'NONE', 'clean', 'BWT')[0]:+.3f}, benign false-alarm rate {V(l, 'NONE', 'clean', 'far_E4')[0]:.3f}.")
out += ["", "## 2. RQ1: how much does a poisoned stage damage each learner?", "Hulk recall lost at E2 (immediate) / persistent damage H at E4, by attack arm (share of Hulk rows poisoned in brackets):"]
for a in nonclean:
    out.append(f"- {a} ({XS[a]:.0%}): " + "; ".join(f"{l} {V(l, 'NONE', a, 'dRec')[0]:.3f} / {V(l, 'NONE', a, 'H')[0]:.3f}" for l in ("FT", "EI", "DR")))
c1 = conf[conf.family == "C1"]
out.append(f"- C1 tests: {int(c1['significant_holm_0.05'].sum())} of {len(c1)} learner-pair comparisons significant after Holm correction (table-confirmatory-tests.csv).")
out += ["", "## 3. RQ2: class-aware admission versus a matched global filter"]
for a in ("NONE", "DC", "DM", "DQ"):
    r = [x for x in at if x[0] == a]
    if r: r = r[0]; out.append(f"- {a} at {strongest}: Hulk recall lost at E2 {r[8]:.3f}, H {r[10]:.3f}, acquisition A {r[12]:.3f}, macro-F1 {r[14]:.3f}; admits {r[2]:.1%} of rows, keeps {r[4]:.1%} of new-class rows, {r[-1]:.1%} of poisoned rows admitted.")
c2 = conf[conf.family == "C2"]
out.append(f"- C2 tests (DC minus DM): {int(c2['significant_holm_0.05'].sum())} of {len(c2)} significant after Holm correction.")
for _, r in c2[c2["significant_holm_0.05"]].iterrows(): out.append(f"  - {r['endpoint']} at {r['arm']}: mean difference {r['mean_diff']:+.3f} (95% CI {r['ci_lo']:+.3f} to {r['ci_hi']:+.3f}), Holm p {r['p_holm']:.4f}.")
out += ["", f"## 4. RQ3: mechanisms and ablations (at {AB_ARM})"]
for r in ab: out.append(f"- {r[0]}: Hulk recall lost at E2 {r[2]:.3f}, H {r[4]:.3f}, macro-F1 {r[8]:.3f} (clean twin {r[-1]:.3f}), poisoned records still in memory {r[16]:.0f}.")
out += ["", "## 5. Cost", ""]
for l in c.index: out.append(f"- {l}: {c.loc[l, 'seconds']:.0f} s per run, {c.loc[l, 'comm_MB']:.1f} MB communicated, {c.loc[l, 'buffer_KB']:.0f} KB replay memory.")
out += ["", "## 6. Limits (state these in the report)",
        "- Development seeds only, one dataset (CICIDS2017-improved), one attack pattern (Hulk relabelled benign in E2, withdrawn afterwards); no final-test data was read.",
        "- The 12 units are 3 ownership partitions (seeds 101-103, the only development seeds the repository allows) times 4 training replicates; replicates inside one partition are not independent of each other, so confidence is bounded by the 3 partitions. Seeds 104 and above stay sealed for the final evaluation.",
        "- The units measure training and ownership variability on a fixed development bank, not variability between captures.",
        "- FT and EI collapse on the clean run (see section 1), so their poisoning damage is bounded by what is left to lose; interpret C1 together with the clean baselines.",
        "- CPU and GPU runs are not bit-identical; all comparisons use runs from one device type.",
        "- The cross-check with the repository's committed results compares two implementations of the same protocol (table-crosscheck-repository.csv); report differences as measured.", ""]
open(f"{OUT3}/analysis-and-discussion.md", "w").write("\n".join(out)); print("\n".join(out))
open(f"{OUT3}/requirement-matrix.md", "w").write("""# Phase 3 requirement matrix
| # | Requirement | Where it is satisfied |
|---|---|---|
| 1 | Implementation of the proposed methodology | Part B: p3lib.py (federated continual learning, FT/EI/DR, 1-5 attackers, DC/DM/DQ admission), p3parallel.py |
| 2 | Dataset and preprocessing | Part A: dataset-classes.csv, preprocessing.md, pinned-hash verification log, per-seed preparation reports |
| 3 | Baseline evaluation | table-baselines.csv, fig1-clean-by-stage, fig2-retention-acquisition |
| 4 | Graphical results | figures/fig1 ... fig7 (png and pdf), each with its source csv |
| 5 | Comparison with existing research | table-literature-mapping.csv, phase2-paper-files.txt, table-crosscheck-repository.csv |
| 6 | Ablation studies | table-ablations.csv, fig6-ablations |
| 7 | Results analysis and discussion | analysis-and-discussion.md, table-dose-response.csv, table-admission.csv, table-confirmatory-tests.csv (C1/C2), table-exploratory-differences.csv |
""")
prov = {os.path.basename(p): hashlib.sha256(open(p, "rb").read()).hexdigest() for p in used}
json.dump(prov, open(f"{OUT3}/run-file-hashes.json", "w"), indent=1)
import zipfile
with zipfile.ZipFile(f"{W}/phase3-part3-outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT3):
        for fn in fs: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), W))
mark("part3_complete"); log("PART C COMPLETE -> /kaggle/working/phase3-part3-outputs.zip")

# Results analysis and discussion

12 paired units (3 ownership partitions x training replicates), paired by unit; confirmatory tests follow the Phase 2 plan (Wilcoxon signed-rank, Holm inside each family). Development bank only. Device: runs_cuda.

## 1. Clean baselines
- FT: final macro-F1 0.031 (sd 0.012), backward transfer -0.719, benign false-alarm rate 0.984.
- EI: final macro-F1 0.046 (sd 0.028), backward transfer -0.688, benign false-alarm rate 0.932.
- DR: final macro-F1 0.491 (sd 0.069), backward transfer -0.047, benign false-alarm rate 0.104.

## 2. RQ1: how much does a poisoned stage damage each learner?
Hulk recall lost at E2 (immediate) / persistent damage H at E4, by attack arm (share of Hulk rows poisoned in brackets):
- k5-d0.25 (25%): FT 0.000 / 0.000; EI 0.000 / 0.000; DR 0.000 / 0.032
- k1-d1 (26%): FT 0.031 / 0.000; EI 0.057 / 0.000; DR 0.130 / 0.162
- k5-d0.5 (50%): FT 0.494 / 0.000; EI 0.530 / 0.000; DR 0.612 / 0.785
- k3-d1 (70%): FT 0.835 / 0.000; EI 0.838 / 0.